In [1]:
import pandas as pd
import numpy as np

import re
import unicodedata
from collections import defaultdict, Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from rapidfuzz import fuzz

ModuleNotFoundError: No module named 'sklearn'

In [2]:
!pip install rapidfuzz scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 6.5 MB/s  0:00:01a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn]0m 3/4 [scikit-learn]

[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

import re
import unicodedata
from collections import defaultdict, Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from rapidfuzz import fuzz

In [4]:
TRAIN_PATH = "../dataset/train/"
TEST_PATH = "../dataset/test/"

s1 = pd.read_csv(
    TRAIN_PATH + "train_source1.tsv",
    sep="\t"
)

s2 = pd.read_csv(
    TRAIN_PATH + "train_source2.tsv",
    sep="\t"
)

s3 = pd.read_csv(
    TRAIN_PATH + "train_source3.tsv",
    sep="\t"
)

ground_truth = pd.read_csv(
    TRAIN_PATH + "train_ground_truth.tsv",
    sep="\t"
)

In [5]:
print("Source 1:", s1.shape)
print("Source 2:", s2.shape)
print("Source 3:", s3.shape)
print("Ground Truth:", ground_truth.shape)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground Truth: (2206821, 2)


In [6]:
display(s1.head())
display(s2.head())
display(s3.head())
display(ground_truth.head())

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [7]:
print("S1 columns:")
print(s1.columns.tolist())

print("\nS2 columns:")
print(s2.columns.tolist())

print("\nS3 columns:")
print(s3.columns.tolist())

print("\nGround truth columns:")
print(ground_truth.columns.tolist())

S1 columns:
['entity_id', 'business_name', 'business_address', 'country']

S2 columns:
['entity_id', 'business_name', 'business_address', 'country']

S3 columns:
['entity_id', 'business_name', 'business_address', 'country']

Ground truth columns:
['source1_entity_id', 'matched_entity_ids']


In [8]:
for name, df in {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3
}.items():
    
    print("\n", "=" * 50)
    print(name)
    print("=" * 50)
    
    print(df.isnull().sum())


Source 1
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

Source 2
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

Source 3
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64


In [9]:
for name, df in {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3
}.items():
    
    print(name)
    print("Duplicate IDs:", df["entity_id"].duplicated().sum())
    print()

Source 1
Duplicate IDs: 0

Source 2
Duplicate IDs: 0

Source 3
Duplicate IDs: 0



In [10]:
def parse_matches(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    
    return [
        item.strip()
        for item in str(x).split(",")
        if item.strip()
    ]

In [11]:
ground_truth["matched_list"] = (
    ground_truth["matched_entity_ids"]
    .apply(parse_matches)
)

ground_truth.head()

,source1_entity_id,matched_entity_ids,matched_list
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...","[S2-681193310, S2-743505751, S3-775321672, S3-..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...","[S2-249013014, S2-197070651, S3-478195123, S3-..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467","[S2-790675320, S2-479876582, S3-878454467]"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215","[S2-153058913, S2-24659151, S3-679606215]"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...","[S2-478959098, S2-553508714, S2-625774905, S3-..."


In [12]:
positive_pairs = []

for _, row in ground_truth.iterrows():
    s1_id = row["source1_entity_id"]
    
    for matched_id in row["matched_list"]:
        positive_pairs.append(
            (s1_id, matched_id)
        )

positive_pairs = pd.DataFrame(
    positive_pairs,
    columns=["source1_entity_id", "matched_entity_id"]
)

positive_pairs.head()

,source1_entity_id,matched_entity_id
0,S1-965667,S2-681193310
1,S1-965667,S2-743505751
2,S1-965667,S3-775321672
3,S1-965667,S3-11291185
4,S1-965667,S3-860443364


In [13]:
print("Number of positive matches:", len(positive_pairs))

Number of positive matches: 7638365


In [14]:
LEGAL_SUFFIXES = {
    "limited": "ltd",
    "ltd": "ltd",
    "corporation": "corp",
    "corporate": "corp",
    "corp": "corp",
    "company": "co",
    "co": "co",
    "private": "pvt",
    "pvt": "pvt",
    "incorporated": "inc",
    "inc": "inc"
}

In [15]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    
    # Unicode normalization
    text = unicodedata.normalize("NFKD", text)
    
    # Remove accents
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    
    # Replace punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [16]:
examples = [
    "ABC Corporation",
    "ABC Corp.",
    "A.B.C. Corporation",
    "  ABC   CORPORATION  "
]

for x in examples:
    print(x, "→", normalize_text(x))

ABC Corporation → abc corporation
ABC Corp. → abc corp
A.B.C. Corporation → a b c corporation
  ABC   CORPORATION   → abc corporation


In [17]:
def preprocess_dataframe(df):
    df = df.copy()
    
    df["name_norm"] = df["business_name"].apply(normalize_text)
    df["address_norm"] = df["business_address"].apply(normalize_text)
    df["country_norm"] = df["country"].apply(normalize_text)
    
    return df

In [19]:
s1 = preprocess_dataframe(s1)
s2 = preprocess_dataframe(s2)
s3 = preprocess_dataframe(s3)

In [20]:
display(
    s1[
        [
            "entity_id",
            "business_name",
            "name_norm",
            "business_address",
            "address_norm",
            "country"
        ]
    ].head(10)
)

,entity_id,business_name,name_norm,business_address,address_norm,country
0,S1-925783039,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US
1,S1-773889195,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US
2,S1-377745466,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US
3,S1-133037285,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US
4,S1-755362802,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India
5,S1-851869949,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India
7,S1-27541239,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US
8,S1-629417405,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US
9,S1-22305073,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US


In [21]:
name_index = defaultdict(list)

for idx, row in s2.iterrows():
    if row["name_norm"]:
        name_index[row["name_norm"]].append(row["entity_id"])

for idx, row in s3.iterrows():
    if row["name_norm"]:
        name_index[row["name_norm"]].append(row["entity_id"])

In [22]:
def name_prefix_key(row):
    return (
        row["country_norm"],
        row["name_norm"][:4]
    )

In [23]:
def get_tokens(text):
    return set(text.split())

In [24]:
def generate_candidates(s1_row, s2, s3):
    candidates = set()
    
    # Block 1
    # Block 2
    # Block 3
    # Block 4
    
    return candidates

In [25]:
candidate_pairs = []

for _, row in s1.iterrows():
    
    candidates = generate_candidates(
        row,
        s2,
        s3
    )
    
    for candidate_id in candidates:
        candidate_pairs.append(
            (
                row["entity_id"],
                candidate_id
            )
        )

In [28]:
from rapidfuzz import fuzz

name1 = "ABC Corporation"
name2 = "ABC Corp"

print("Ratio:", fuzz.ratio(name1, name2))
print("Partial Ratio:", fuzz.partial_ratio(name1, name2))
print("Token Sort Ratio:", fuzz.token_sort_ratio(name1, name2))
print("Token Set Ratio:", fuzz.token_set_ratio(name1, name2))

Ratio: 69.56521739130434
Partial Ratio: 100.0
Token Sort Ratio: 69.56521739130434
Token Set Ratio: 69.56521739130434


In [29]:
print("Source 1 shape:", s1.shape)
print("Source 2 shape:", s2.shape)
print("Source 3 shape:", s3.shape)
print("Ground truth shape:", ground_truth.shape)

Source 1 shape: (2206821, 7)
Source 2 shape: (5034616, 7)
Source 3 shape: (5285603, 7)
Ground truth shape: (2206821, 3)


In [30]:
display(s1.head())
display(s2.head())
display(s3.head())
display(ground_truth.head())

,entity_id,business_name,business_address,country,name_norm,address_norm,country_norm
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester drive high point nc,us
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis road tahlequah ok,us
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail inc,1712 montebello avenue phoenix az,us
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron drive unit apartment g dundalk md,us
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal,india


,entity_id,business_name,business_address,country,name_norm,address_norm,country_norm
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,,kh no 570 13 new delhi west delhi delhi,india
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,holloway peak inc seafood,105 elm st morganton nc,us
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,,g 3 571 gulmohar colony bhopal madhya pradesh,india
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,summit inc,greensboro nc 19 1 2 stardust trail,us
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,delta tetlecommunication inc,914 pierpont ave cleveland oh,us


,entity_id,business_name,business_address,country,name_norm,address_norm,country_norm
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US,wilfordhancock com,mack rd haltom city texas,us
1,S3-859268022,International South Consultants Private Ltd,NaN,India,international south consultants private ltd,,india
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US,llc moncada learning center,5780 fawn ct fort worth texas,us
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US,moyna s coffee,1 ivanhoe ave po box 6009 cincinnati ohio,us
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India,pvt efs print ventures ltd,door no 183 41st cross 22nd main 9th block jay...,india


,source1_entity_id,matched_entity_ids,matched_list
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...","[S2-681193310, S2-743505751, S3-775321672, S3-..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...","[S2-249013014, S2-197070651, S3-478195123, S3-..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467","[S2-790675320, S2-479876582, S3-878454467]"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215","[S2-153058913, S2-24659151, S3-679606215]"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...","[S2-478959098, S2-553508714, S2-625774905, S3-..."


In [31]:
print("S1 missing values:")
print(s1.isnull().sum())

print("\nS2 missing values:")
print(s2.isnull().sum())

print("\nS3 missing values:")
print(s3.isnull().sum())

S1 missing values:
entity_id           0
business_name       0
business_address    0
country             0
name_norm           0
address_norm        0
country_norm        0
dtype: int64

S2 missing values:
entity_id                0
business_name            2
business_address    168967
country                  0
name_norm                0
address_norm             0
country_norm             0
dtype: int64

S3 missing values:
entity_id                0
business_name           13
business_address    175916
country                  0
name_norm                0
address_norm             0
country_norm             0
dtype: int64


In [32]:
print("=" * 50)
print("COUNTRY DISTRIBUTION")
print("=" * 50)

print("\nSource 1:")
print(s1["country"].value_counts(dropna=False))

print("\nSource 2:")
print(s2["country"].value_counts(dropna=False))

print("\nSource 3:")
print(s3["country"].value_counts(dropna=False))

COUNTRY DISTRIBUTION

Source 1:
country
US       1323633
India     883188
Name: count, dtype: int64

Source 2:
country
US       3016817
India    2017799
Name: count, dtype: int64

Source 3:
country
US       3170056
India    2115547
Name: count, dtype: int64


In [33]:
print("\nRecord counts:")
print("S1:", len(s1))
print("S2:", len(s2))
print("S3:", len(s3))


Record counts:
S1: 2206821
S2: 5034616
S3: 5285603


In [34]:
print("=" * 50)
print("DUPLICATE ANALYSIS")
print("=" * 50)

for name, df in [("S1", s1), ("S2", s2), ("S3", s3)]:
    print(f"\n{name}")
    
    print("Duplicate entity IDs:",
          df["entity_id"].duplicated().sum())
    
    print("Duplicate business names:",
          df["name_norm"].duplicated().sum())
    
    print("Duplicate addresses:",
          df["address_norm"].duplicated().sum())

DUPLICATE ANALYSIS

S1
Duplicate entity IDs: 0
Duplicate business names: 686137
Duplicate addresses: 76649

S2
Duplicate entity IDs: 0
Duplicate business names: 1288643
Duplicate addresses: 753243

S3
Duplicate entity IDs: 0
Duplicate business names: 1243756
Duplicate addresses: 671057


In [35]:
ground_truth["matched_list"] = (
    ground_truth["matched_entity_ids"]
    .fillna("")
    .apply(
        lambda x: [
            item.strip()
            for item in str(x).split(",")
            if item.strip()
        ]
    )
)

ground_truth["match_count"] = ground_truth["matched_list"].apply(len)

print(ground_truth["match_count"].describe())

count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: match_count, dtype: float64


In [36]:
print("\nMatch count distribution:")
print(ground_truth["match_count"].value_counts().sort_index())


Match count distribution:
match_count
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64


In [37]:
singletons = (ground_truth["match_count"] == 0).sum()

print("Total S1 entities:", len(ground_truth))
print("Singleton S1 entities:", singletons)
print(
    "Singleton percentage:",
    round(singletons / len(ground_truth) * 100, 2),
    "%"
)

Total S1 entities: 2206821
Singleton S1 entities: 123247
Singleton percentage: 5.58 %


In [38]:
print("=" * 50)
print("COUNTRY DISTRIBUTION")
print("=" * 50)

print("\nSource 1:")
print(s1["country"].value_counts(dropna=False))

print("\nSource 2:")
print(s2["country"].value_counts(dropna=False))

print("\nSource 3:")
print(s3["country"].value_counts(dropna=False))

COUNTRY DISTRIBUTION

Source 1:
country
US       1323633
India     883188
Name: count, dtype: int64

Source 2:
country
US       3016817
India    2017799
Name: count, dtype: int64

Source 3:
country
US       3170056
India    2115547
Name: count, dtype: int64


In [39]:
print("=" * 50)
print("DUPLICATE ANALYSIS")
print("=" * 50)

for name, df in [("S1", s1), ("S2", s2), ("S3", s3)]:
    print(f"\n{name}")
    
    print("Duplicate entity IDs:",
          df["entity_id"].duplicated().sum())
    
    print("Duplicate normalized names:",
          df["name_norm"].duplicated().sum())
    
    print("Duplicate normalized addresses:",
          df["address_norm"].duplicated().sum())

DUPLICATE ANALYSIS

S1
Duplicate entity IDs: 0
Duplicate normalized names: 686137
Duplicate normalized addresses: 76649

S2
Duplicate entity IDs: 0
Duplicate normalized names: 1288643
Duplicate normalized addresses: 753243

S3
Duplicate entity IDs: 0
Duplicate normalized names: 1243756
Duplicate normalized addresses: 671057


In [40]:
print("=" * 50)
print("MATCH DISTRIBUTION")
print("=" * 50)

print(ground_truth["match_count"].describe())

MATCH DISTRIBUTION
count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: match_count, dtype: float64


In [41]:
print("\nMatch count distribution:")
print(
    ground_truth["match_count"]
    .value_counts()
    .sort_index()
)


Match count distribution:
match_count
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64


In [42]:
candidate_pairs = []

for _, row in s1.iterrows():
    
    candidates = generate_candidates(
        row,
        s2,
        s3
    )
    
    for candidate_id in candidates:
        candidate_pairs.append(
            (
                row["entity_id"],
                candidate_id
            )
        )

In [43]:
fuzz.ratio(name1, name2)
fuzz.partial_ratio(name1, name2)
fuzz.token_sort_ratio(name1, name2)
fuzz.token_set_ratio(name1, name2)

69.56521739130434

In [45]:
from rapidfuzz import fuzz

address1 = "123 Main Road, Delhi"
address2 = "123 Main Rd Delhi"

print("Ratio:", fuzz.ratio(address1, address2))
print("Partial Ratio:", fuzz.partial_ratio(address1, address2))
print("Token Sort Ratio:", fuzz.token_sort_ratio(address1, address2))
print("Token Set Ratio:", fuzz.token_set_ratio(address1, address2))

Ratio: 91.89189189189189
Partial Ratio: 82.35294117647058
Token Sort Ratio: 91.89189189189189
Token Set Ratio: 91.89189189189189


In [46]:
address1 = s1.loc[0, "address_norm"]
address2 = s2.loc[0, "address_norm"]

print(fuzz.ratio(address1, address2))

40.0


In [47]:
fuzz.ratio(address1, address2)

40.0

In [48]:
%whos

Variable               Type           Data/Info
-----------------------------------------------
Counter                type           <class 'collections.Counter'>
LEGAL_SUFFIXES         dict           n=11
TEST_PATH              str            ../dataset/test/
TRAIN_PATH             str            ../dataset/train/
TfidfVectorizer        type           <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
address1               str            1795 westchester drive high point nc
address2               str            kh no 570 13 new delhi west delhi delhi
candidate_pairs        list           n=0
candidates             set            set()
defaultdict            type           <class 'collections.defaultdict'>
df                     DataFrame      Shape: (5285603, 7)
examples               list           n=4
fuzz                   module         <module 'rapidfuzz.fuzz' <...>kages/rapidfuzz/fuzz.py'>
generate_candidates    function       <function generate_candidates at 0x1194abab0>

In [49]:
generate_candidates()

TypeError: generate_candidates() missing 3 required positional arguments: 's1_row', 's2', and 's3'

In [50]:
generate_candidates(s1_row, s2, s3)

NameError: name 's1_row' is not defined

In [51]:
s1_row = s1.iloc[0]

print("S1 ID:", s1_row["entity_id"])
print("Name:", s1_row["business_name"])
print("Address:", s1_row["business_address"])
print("Country:", s1_row["country"])

S1 ID: S1-925783039
Name: Orelee's Barbershop
Address: 1795 Westchester Drive, High Point, NC
Country: US


In [52]:
test_candidates = generate_candidates(
    s1_row,
    s2,
    s3
)

print("Number of candidates:", len(test_candidates))
print("Candidates:", list(test_candidates)[:20])

Number of candidates: 0
Candidates: []


In [53]:
import inspect

print(inspect.getsource(generate_candidates))

def generate_candidates(s1_row, s2, s3):
    candidates = set()

    # Block 1
    # Block 2
    # Block 3
    # Block 4

    return candidates



In [54]:
def generate_candidates(s1_row, s2, s3):
    candidates = set()

    name = s1_row["name_norm"]
    address = s1_row["address_norm"]
    country = s1_row["country_norm"]

    # -----------------------------
    # BLOCK 1: Exact normalized name
    # -----------------------------
    if name:
        key = (country, name)

        if key in name_index:
            candidates.update(name_index[key])

    # -----------------------------
    # BLOCK 2: Name prefix
    # -----------------------------
    if name:
        prefix = name[:4]

        if prefix in name_prefix_index:
            candidates.update(name_prefix_index[prefix])

    # -----------------------------
    # BLOCK 3: Address tokens
    # -----------------------------
    if address:
        tokens = address.split()

        for token in tokens:
            if len(token) >= 4:
                key = (country, token)

                if key in address_index:
                    candidates.update(address_index[key])

    # -----------------------------
    # BLOCK 4: Limit extremely large
    # blocks
    # -----------------------------
    if len(candidates) > 500:
        candidates = set(list(candidates)[:500])

    return candidates

In [55]:
from collections import defaultdict

name_index = defaultdict(list)
name_prefix_index = defaultdict(list)
address_index = defaultdict(list)

# ==========================================
# S2 + S3 combined indexing
# ==========================================

for df in [s2, s3]:

    for _, row in df.iterrows():

        entity_id = row["entity_id"]
        country = row["country_norm"]
        name = row["name_norm"]
        address = row["address_norm"]

        # -----------------------------
        # Exact name index
        # -----------------------------
        if name:
            key = (country, name)
            name_index[key].append(entity_id)

            # -----------------------------
            # Name prefix index
            # -----------------------------
            prefix = name[:4]

            if len(prefix) >= 3:
                name_prefix_index[prefix].append(entity_id)

        # -----------------------------
        # Address token index
        # -----------------------------
        if address:

            tokens = set(address.split())

            for token in tokens:

                if len(token) >= 4:
                    key = (country, token)
                    address_index[key].append(entity_id)

print("Name index:", len(name_index))
print("Name prefix index:", len(name_prefix_index))
print("Address index:", len(address_index))

Name index: 7165488
Name prefix index: 139570
Address index: 758056


In [56]:
def generate_candidates(s1_row, s2, s3):

    candidates = set()

    name = s1_row["name_norm"]
    address = s1_row["address_norm"]
    country = s1_row["country_norm"]

    # ==========================================
    # BLOCK 1 — Exact Name + Country
    # ==========================================

    if name:
        key = (country, name)

        if key in name_index:
            candidates.update(name_index[key])

    # ==========================================
    # BLOCK 2 — Name Prefix
    # ==========================================

    if name:
        prefix = name[:4]

        if prefix in name_prefix_index:
            candidates.update(name_prefix_index[prefix])

    # ==========================================
    # BLOCK 3 — Address Token + Country
    # ==========================================

    if address:

        for token in set(address.split()):

            if len(token) >= 4:

                key = (country, token)

                if key in address_index:
                    candidates.update(address_index[key])

    return candidates

In [57]:
s1_row = s1.iloc[0]

test_candidates = generate_candidates(
    s1_row,
    s2,
    s3
)

print("S1 ID:", s1_row["entity_id"])
print("Business:", s1_row["business_name"])
print("Country:", s1_row["country"])

print("\nNumber of candidates:", len(test_candidates))
print("First 20 candidates:", list(test_candidates)[:20])

S1 ID: S1-925783039
Business: Orelee's Barbershop
Country: US

Number of candidates: 498940
First 20 candidates: ['S3-797632358', 'S2-663904690', 'S3-309201536', 'S3-115365312', 'S2-82302861', 'S3-369740542', 'S3-910019768', 'S2-881407300', 'S3-909269883', 'S2-531770578', 'S2-23706941', 'S3-167477811', 'S3-156965591', 'S3-837604914', 'S3-491571578', 'S2-693683929', 'S3-366903850', 'S2-968821279', 'S3-793043181', 'S2-582885097']


In [58]:
s1_id = s1_row["entity_id"]

true_matches = set(
    ground_truth.loc[
        ground_truth["source1_entity_id"] == s1_id,
        "matched_list"
    ].iloc[0]
)

print("True matches:", true_matches)

print("Number of true matches:", len(true_matches))

print("True matches recovered:",
      true_matches.issubset(test_candidates))

True matches: {'S2-517291332', 'S2-157377754', 'S3-997698194', 'S3-698172821'}
Number of true matches: 4
True matches recovered: True


In [59]:
import os

print("matching_results.tsv exists:",
      os.path.exists("../output/matching_results.tsv"))

print("candidate_pairs.tsv exists:",
      os.path.exists("../output/candidate_pairs.tsv"))

matching_results.tsv exists: False
candidate_pairs.tsv exists: False


In [60]:
import os
print(os.getcwd())

/Users/vikastiwari/ML_Challenge_2026/notebooks


In [61]:
test_s1 = pd.read_csv(
    "../dataset/test/test_source1.tsv",
    sep="\t"
)

test_s2 = pd.read_csv(
    "../dataset/test/test_source2.tsv",
    sep="\t"
)

test_s3 = pd.read_csv(
    "../dataset/test/test_source3.tsv",
    sep="\t"
)

test_s1 = preprocess_dataframe(test_s1)
test_s2 = preprocess_dataframe(test_s2)
test_s3 = preprocess_dataframe(test_s3)

print("Test S1:", test_s1.shape)
print("Test S2:", test_s2.shape)
print("Test S3:", test_s3.shape)

Test S1: (1732544, 7)
Test S2: (4887273, 7)
Test S3: (4771003, 7)


In [62]:
import os

os.makedirs("../output", exist_ok=True)

print("Output directory ready:", os.path.abspath("../output"))

Output directory ready: /Users/vikastiwari/ML_Challenge_2026/output


In [63]:
print("Test S1:", test_s1.shape)
print("Test S2:", test_s2.shape)
print("Test S3:", test_s3.shape)

Test S1: (1732544, 7)
Test S2: (4887273, 7)
Test S3: (4771003, 7)


In [64]:
from collections import defaultdict

name_index = defaultdict(list)
prefix_index = defaultdict(list)
address_index = defaultdict(list)

# -----------------------------
# Index S2
# -----------------------------
for row in test_s2.itertuples(index=False):

    entity_id = row.entity_id
    country = row.country_norm
    name = row.name_norm
    address = row.address_norm

    if name:
        name_index[(country, name)].append(entity_id)

        prefix = name[:4]
        if len(prefix) >= 3:
            prefix_index[(country, prefix)].append(entity_id)

    if address:
        for token in set(address.split()):
            if len(token) >= 4:
                address_index[(country, token)].append(entity_id)


# -----------------------------
# Index S3
# -----------------------------
for row in test_s3.itertuples(index=False):

    entity_id = row.entity_id
    country = row.country_norm
    name = row.name_norm
    address = row.address_norm

    if name:
        name_index[(country, name)].append(entity_id)

        prefix = name[:4]
        if len(prefix) >= 3:
            prefix_index[(country, prefix)].append(entity_id)

    if address:
        for token in set(address.split()):
            if len(token) >= 4:
                address_index[(country, token)].append(entity_id)

print("Exact name blocks:", len(name_index))
print("Name prefix blocks:", len(prefix_index))
print("Address token blocks:", len(address_index))

Exact name blocks: 6872632
Name prefix blocks: 189778
Address token blocks: 757371


In [65]:
def generate_test_candidates(row):
    
    candidates = set()

    country = row.country_norm
    name = row.name_norm
    address = row.address_norm

    # Block 1: exact name + country
    if name:
        candidates.update(
            name_index.get((country, name), [])
        )

    # Block 2: name prefix + country
    if name:
        candidates.update(
            prefix_index.get((country, name[:4]), [])
        )

    # Block 3: address tokens + country
    if address:
        for token in set(address.split()):
            if len(token) >= 4:
                candidates.update(
                    address_index.get((country, token), [])
                )

    return candidates

In [66]:
for i in range(10):

    row = test_s1.iloc[i]

    candidates = generate_test_candidates(row)

    print(
        i,
        row["entity_id"],
        "=>",
        len(candidates),
        "candidates"
    )

0 S1-714132312 => 322970 candidates
1 S1-106407869 => 389179 candidates
2 S1-156285671 => 332751 candidates
3 S1-689823050 => 481608 candidates
4 S1-921369899 => 205460 candidates
5 S1-481669221 => 732709 candidates
6 S1-909865979 => 22221 candidates
7 S1-280204013 => 68045 candidates
8 S1-742053041 => 984463 candidates
9 S1-913506265 => 245013 candidates


In [67]:
for token in set(address.split()):
    candidates.update(address_index.get((country, token), []))

In [68]:
from collections import Counter

address_token_freq = Counter()

for df in [test_s2, test_s3]:

    for address in df["address_norm"]:

        if address:
            tokens = set(address.split())

            for token in tokens:
                if len(token) >= 4:
                    address_token_freq[token] += 1

print("Total unique address tokens:", len(address_token_freq))

print("\nMost common tokens:")
print(address_token_freq.most_common(20))

Total unique address tokens: 713253

Most common tokens:
[('road', 1170431), ('floor', 656551), ('nagar', 612602), ('delhi', 591147), ('city', 512926), ('street', 480941), ('mumbai', 401207), ('west', 394792), ('maharashtra', 388746), ('plot', 376583), ('avenue', 347997), ('north', 307625), ('pradesh', 291438), ('drive', 275979), ('bangalore', 269905), ('door', 258957), ('loire', 252964), ('bordeaux', 226118), ('block', 222441), ('near', 221323)]


In [69]:
for threshold in [10, 50, 100, 500, 1000]:

    count = sum(
        1 for freq in address_token_freq.values()
        if freq <= threshold
    )

    print(
        f"Tokens appearing <= {threshold} times:",
        count
    )

Tokens appearing <= 10 times: 568567
Tokens appearing <= 50 times: 664169
Tokens appearing <= 100 times: 683386
Tokens appearing <= 500 times: 705499
Tokens appearing <= 1000 times: 709296


In [71]:
# Check what ML/model variables currently exist
%whos

Variable                   Type           Data/Info
---------------------------------------------------
Counter                    type           <class 'collections.Counter'>
LEGAL_SUFFIXES             dict           n=11
TEST_PATH                  str            ../dataset/test/
TRAIN_PATH                 str            ../dataset/train/
TfidfVectorizer            type           <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
address                    str            
address1                   str            1795 westchester drive high point nc
address2                   str            kh no 570 13 new delhi west delhi delhi
address_index              defaultdict    defaultdict(<class 'list'<...>one'): ['S3-746949712']})
address_token_freq         Counter        Counter({'road': 1170431,<...>nn': 1, 'briarscone': 1})
candidate_pairs            list           n=0
candidates                 set            {'S3-764088105', 'S3-4163<...>4032542', 'S3-714929039'}
count           

IOStream.flush timed out


In [72]:
[x for x in dir() if any(
    word in x.lower()
    for word in [
        "model", "classifier", "clf",
        "threshold", "feature", "train"
    ]
)]

['TRAIN_PATH', 'threshold', 'train_test_split']

In [73]:
print("S1:", [c for c in s1.columns if "_norm" in c])
print("S2:", [c for c in s2.columns if "_norm" in c])
print("S3:", [c for c in s3.columns if "_norm" in c])

print("\nTest S1:", [c for c in test_s1.columns if "_norm" in c])
print("Test S2:", [c for c in test_s2.columns if "_norm" in c])
print("Test S3:", [c for c in test_s3.columns if "_norm" in c])

S1: ['name_norm', 'address_norm', 'country_norm']
S2: ['name_norm', 'address_norm', 'country_norm']
S3: ['name_norm', 'address_norm', 'country_norm']

Test S1: ['name_norm', 'address_norm', 'country_norm']
Test S2: ['name_norm', 'address_norm', 'country_norm']
Test S3: ['name_norm', 'address_norm', 'country_norm']


In [74]:
sample = s1[
    ["business_name", "name_norm",
     "business_address", "address_norm",
     "country", "country_norm"]
].head(10)

display(sample)

,business_name,name_norm,business_address,address_norm,country,country_norm
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,US,us
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,US,us
2,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,US,us
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,US,us
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,India,india
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,US,us
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,India,india
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,US,us
8,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,US,us
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,US,us


In [75]:
print(
    "Uppercase characters remaining:",
    s1["name_norm"].str.contains(r"[A-Z]", regex=True).sum()
)

Uppercase characters remaining: 0


In [76]:
for name, df in [
    ("S1", s1),
    ("S2", s2),
    ("S3", s3),
    ("Test S1", test_s1),
    ("Test S2", test_s2),
    ("Test S3", test_s3)
]:
    
    uppercase = df["name_norm"].str.contains(
        r"[A-Z]",
        regex=True,
        na=False
    ).sum()
    
    print(name, "uppercase:", uppercase)

S1 uppercase: 0
S2 uppercase: 0
S3 uppercase: 0
Test S1 uppercase: 0
Test S2 uppercase: 0
Test S3 uppercase: 0


In [77]:
re.sub(r"[^a-z0-9\s]", " ", text)

NameError: name 'text' is not defined

In [78]:
re.sub(r"[^a-z0-9\s]", " ", text)

NameError: name 'text' is not defined

In [79]:
for name, df in [
    ("S1", s1),
    ("S2", s2),
    ("S3", s3),
    ("Test S1", test_s1),
    ("Test S2", test_s2),
    ("Test S3", test_s3)
]:

    bad_names = df["name_norm"].str.contains(
        r"[^a-z0-9\s]",
        regex=True,
        na=False
    ).sum()

    bad_addresses = df["address_norm"].str.contains(
        r"[^a-z0-9\s]",
        regex=True,
        na=False
    ).sum()

    uppercase_names = df["name_norm"].str.contains(
        r"[A-Z]",
        regex=True,
        na=False
    ).sum()

    print(
        f"{name}: "
        f"bad name chars={bad_names}, "
        f"bad address chars={bad_addresses}, "
        f"uppercase names={uppercase_names}"
    )

S1: bad name chars=0, bad address chars=0, uppercase names=0
S2: bad name chars=0, bad address chars=0, uppercase names=0
S3: bad name chars=0, bad address chars=0, uppercase names=0
Test S1: bad name chars=0, bad address chars=0, uppercase names=0
Test S2: bad name chars=0, bad address chars=0, uppercase names=0
Test S3: bad name chars=0, bad address chars=0, uppercase names=0


In [80]:
display(
    test_s1[
        [
            "business_name",
            "name_norm",
            "business_address",
            "address_norm",
            "country",
            "country_norm"
        ]
    ].head(10)
)

,business_name,name_norm,business_address,address_norm,country,country_norm
0,Zephay Labs Inc,zephay labs inc,"2621 Cotten Road, Tyler, TX",2621 cotten road tyler tx,US,us
1,Vision Partners Corp,vision partners corp,"IA, Iowa City, 1064 Newton Rd, Unit 11",ia iowa city 1064 newton rd unit 11,US,us
2,<< Team Ecole,team ecole,"175 Boulevard du Président Franklin Roosevelt,...",175 boulevard du president franklin roosevelt ...,France,france
3,Red Perfect Trading,red perfect trading,"Mirzapur, Ews 12, Uttar Pradesh, Mirzapursadar...",mirzapur ews 12 uttar pradesh mirzapursadar aw...,India,india
4,ZNB Club SARL,znb club sarl,"Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Ru...",nouvelle aquitaine la teste de buch 5 bis rue ...,France,france
5,Nandlal Kisan LLP,nandlal kisan llp,"D-61 Ifs Apartmentmayur Vihar I, New Delhi, Ea...",d 61 ifs apartmentmayur vihar i new delhi east...,India,india
6,Cure Seafood,cure seafood,"1325 Brooklyn Walk, Issaquah, WA",1325 brooklyn walk issaquah wa,US,us
7,Om Constructions Pvt Ltd,om constructions pvt ltd,"Karauli, Rajasthan, Karauli, Pani Ki Tanki Ke ...",karauli rajasthan karauli pani ki tanki ke pas...,India,india
8,Roongta Sangh,roongta sangh,"Bhubaneswar, Sub Plot No.-L6/29, Mahodadhi Bha...",bhubaneswar sub plot no l6 29 mahodadhi bhawan...,India,india
9,Thermal & Fils SASU,thermal fils sasu,"20 Rue Parmentier, Dunkerque, Hauts-de-France",20 rue parmentier dunkerque hauts de france,France,france


In [98]:
from collections import defaultdict, Counter

train_name_index = defaultdict(list)
train_address_index = defaultdict(list)

for df in [s2, s3]:

    for row in df.itertuples(index=False):

        entity_id = row.entity_id
        country = row.country_norm
        name = row.name_norm
        address = row.address_norm

        # Exact name + country
        if name:
            train_name_index[
                (country, name)
            ].append(entity_id)

        # Address tokens
        if address:

            for token in set(address.split()):

                if len(token) >= 4:
                    train_address_index[
                        (country, token)
                    ].append(entity_id)

print("Name blocks:", len(train_name_index))
print("Address blocks:", len(train_address_index))

Name blocks: 7165488
Address blocks: 758056


In [99]:
address_freq = Counter()

for df in [s2, s3]:

    for address in df["address_norm"]:

        if address:

            for token in set(address.split()):

                if len(token) >= 4:
                    address_freq[token] += 1

print("Unique address tokens:", len(address_freq))

print("\nMost common:")
print(address_freq.most_common(20))

Unique address tokens: 721890

Most common:
[('road', 1288416), ('street', 694453), ('city', 650527), ('floor', 597331), ('nagar', 554778), ('delhi', 535355), ('drive', 457748), ('avenue', 405843), ('north', 399761), ('west', 392303), ('mumbai', 360514), ('maharashtra', 343014), ('plot', 341980), ('texas', 294859), ('null', 262638), ('pradesh', 258312), ('lane', 243506), ('bangalore', 240821), ('york', 234392), ('door', 221837)]


In [100]:
def get_training_candidates(s1_row):

    candidates = set()

    country = s1_row["country_norm"]
    name = s1_row["name_norm"]
    address = s1_row["address_norm"]

    # =================================
    # Block A: exact name + country
    # =================================

    if name:

        candidates.update(
            train_name_index.get(
                (country, name),
                []
            )
        )

    # =================================
    # Block B: rare address tokens
    # =================================

    if address:

        tokens = [
            token
            for token in set(address.split())
            if len(token) >= 4
        ]

        # rarest tokens first
        tokens = sorted(
            tokens,
            key=lambda x: address_freq.get(
                x,
                float("inf")
            )
        )

        # Only rare tokens
        for token in tokens[:3]:

            frequency = address_freq.get(
                token,
                0
            )

            if 0 < frequency <= 500:

                candidates.update(
                    train_address_index.get(
                        (country, token),
                        []
                    )
                )

    return candidates

In [101]:
total_matches = 0
recovered_matches = 0

for i in range(1000):

    row = s1.iloc[i]

    s1_id = row["entity_id"]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"] == s1_id,
            "matched_list"
        ].iloc[0]
    )

    candidates = get_training_candidates(row)

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(candidates)
    )

print("Total true matches:", total_matches)
print("Recovered true matches:", recovered_matches)

if total_matches > 0:
    print(
        "Candidate Recall:",
        recovered_matches / total_matches
    )

Total true matches: 3517
Recovered true matches: 2301
Candidate Recall: 0.6542507819164061


In [104]:
train_name_token_index = defaultdict(list)
name_token_freq = Counter()

for df in [s2, s3]:

    for row in df.itertuples(index=False):

        entity_id = row.entity_id
        country = row.country_norm
        name = row.name_norm

        if not name:
            continue

        tokens = set(
            token
            for token in name.split()
            if len(token) >= 3
        )

        for token in tokens:

            key = (country, token)

            train_name_token_index[key].append(
                entity_id
            )

            name_token_freq[key] += 1

print(
    "Name token blocks:",
    len(train_name_token_index)
)

Name token blocks: 1480028


In [107]:
def get_training_candidates_v2(s1_row):

    candidates = set()

    country = s1_row["country_norm"]
    name = s1_row["name_norm"]
    address = s1_row["address_norm"]

    # ==========================================
    # BLOCK 1 — Exact name + country
    # ==========================================

    if name:

        candidates.update(
            train_name_index.get(
                (country, name),
                []
            )
        )

    # ==========================================
    # BLOCK 2 — Name token matching
    # ==========================================

    if name:

        name_tokens = [
            token
            for token in set(name.split())
            if len(token) >= 3
        ]

        # rarest name tokens first
        name_tokens = sorted(
            name_tokens,
            key=lambda x: name_token_freq.get(
                (country, x),
                float("inf")
            )
        )

        for token in name_tokens[:2]:

            frequency = name_token_freq.get(
                (country, token),
                0
            )

            # avoid extremely common tokens
            if 0 < frequency <= 1000:

                candidates.update(
                    train_name_token_index.get(
                        (country, token),
                        []
                    )
                )

    # ==========================================
    # BLOCK 3 — Rare address tokens
    # ==========================================

    if address:

        address_tokens = [
            token
            for token in set(address.split())
            if len(token) >= 4
        ]

        address_tokens = sorted(
            address_tokens,
            key=lambda x: address_freq.get(
                x,
                float("inf")
            )
        )

        for token in address_tokens[:3]:

            frequency = address_freq.get(
                token,
                0
            )

            if 0 < frequency <= 500:

                candidates.update(
                    train_address_index.get(
                        (country, token),
                        []
                    )
                )

    return candidates

In [109]:
total_matches = 0
recovered_matches = 0

for i in range(1000):

    row = s1.iloc[i]

    s1_id = row["entity_id"]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"] == s1_id,
            "matched_list"
        ].iloc[0]
    )

    candidates = get_training_candidates_v2(row)

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(candidates)
    )

print("Total true matches:", total_matches)
print("Recovered true matches:", recovered_matches)

print(
    "Candidate Recall:",
    recovered_matches / total_matches
)

Total true matches: 3517
Recovered true matches: 2712
Candidate Recall: 0.7711117429627523


In [112]:
train_chargram_index = defaultdict(list)
chargram_freq = Counter()

def get_chargrams(text, n=3):
    if not text:
        return set()

    text = text.replace(" ", "")
    
    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


for df in [s2, s3]:

    for row in df.itertuples(index=False):

        entity_id = row.entity_id
        country = row.country_norm
        name = row.name_norm

        if not name:
            continue

        grams = get_chargrams(name, 3)

        for gram in grams:

            key = (country, gram)

            train_chargram_index[key].append(
                entity_id
            )

            chargram_freq[key] += 1


print("Character gram blocks:",
      len(train_chargram_index))

Character gram blocks: 68090


In [114]:
def get_training_candidates_v3(s1_row):

    candidates = set()

    country = s1_row["country_norm"]
    name = s1_row["name_norm"]
    address = s1_row["address_norm"]

    # ==========================================
    # BLOCK 1 — Exact name
    # ==========================================

    if name:

        candidates.update(
            train_name_index.get(
                (country, name),
                []
            )
        )

    # ==========================================
    # BLOCK 2 — Name token
    # ==========================================

    if name:

        tokens = [
            token
            for token in set(name.split())
            if len(token) >= 3
        ]

        tokens = sorted(
            tokens,
            key=lambda x: name_token_freq.get(
                (country, x),
                float("inf")
            )
        )

        for token in tokens[:2]:

            frequency = name_token_freq.get(
                (country, token),
                0
            )

            if 0 < frequency <= 1000:

                candidates.update(
                    train_name_token_index.get(
                        (country, token),
                        []
                    )
                )

    # ==========================================
    # BLOCK 3 — Character 3-grams
    # ==========================================

    if name:

        grams = get_chargrams(name, 3)

        # Select rarest 3 grams
        grams = sorted(
            grams,
            key=lambda x: chargram_freq.get(
                (country, x),
                float("inf")
            )
        )

        for gram in grams[:3]:

            frequency = chargram_freq.get(
                (country, gram),
                0
            )

            if 0 < frequency <= 300:

                candidates.update(
                    train_chargram_index.get(
                        (country, gram),
                        []
                    )
                )

    # ==========================================
    # BLOCK 4 — Rare address tokens
    # ==========================================

    if address:

        tokens = [
            token
            for token in set(address.split())
            if len(token) >= 4
        ]

        tokens = sorted(
            tokens,
            key=lambda x: address_freq.get(
                x,
                float("inf")
            )
        )

        for token in tokens[:3]:

            frequency = address_freq.get(
                token,
                0
            )

            if 0 < frequency <= 500:

                candidates.update(
                    train_address_index.get(
                        (country, token),
                        []
                    )
                )

    return candidates

In [115]:
total_matches = 0
recovered_matches = 0

for i in range(1000):

    row = s1.iloc[i]

    s1_id = row["entity_id"]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"] == s1_id,
            "matched_list"
        ].iloc[0]
    )

    candidates = get_training_candidates_v3(row)

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(candidates)
    )

print("Total true matches:", total_matches)
print("Recovered true matches:", recovered_matches)

print(
    "Candidate Recall:",
    recovered_matches / total_matches
)

Total true matches: 3517
Recovered true matches: 2743
Candidate Recall: 0.7799260733579756


In [116]:
train_char2_index = defaultdict(list)
char2_freq = Counter()

for df in [s2, s3]:

    for row in df.itertuples(index=False):

        entity_id = row.entity_id
        country = row.country_norm
        name = row.name_norm

        if not name:
            continue

        grams = get_chargrams(name, 2)

        for gram in grams:

            key = (country, gram)

            train_char2_index[key].append(entity_id)
            char2_freq[key] += 1

print("2-gram blocks:", len(train_char2_index))

2-gram blocks: 2581


In [119]:
train_name_start_index = defaultdict(list)

for df in [s2, s3]:

    for row in df.itertuples(index=False):

        entity_id = row.entity_id
        country = row.country_norm
        name = row.name_norm

        if not name:
            continue

        compact = name.replace(" ", "")

        if len(compact) >= 3:

            key = (
                country,
                compact[:3]
            )

            train_name_start_index[key].append(
                entity_id
            )

print(
    "Name-start blocks:",
    len(train_name_start_index)
)

Name-start blocks: 46688


In [120]:
train_name_start_index = defaultdict(list)

for df in [s2, s3]:

    for row in df.itertuples(index=False):

        entity_id = row.entity_id
        country = row.country_norm
        name = row.name_norm

        if not name:
            continue

        compact = name.replace(" ", "")

        if len(compact) >= 3:

            key = (
                country,
                compact[:3]
            )

            train_name_start_index[key].append(
                entity_id
            )

print(
    "Name-start blocks:",
    len(train_name_start_index)
)

Name-start blocks: 46688


In [121]:
train_number_index = defaultdict(list)
number_freq = Counter()

for df in [s2, s3]:

    for row in df.itertuples(index=False):

        entity_id = row.entity_id
        country = row.country_norm
        address = row.address_norm

        if not address:
            continue

        numbers = set(
            re.findall(r"\b\d{2,6}\b", address)
        )

        for number in numbers:

            key = (country, number)

            train_number_index[key].append(
                entity_id
            )

            number_freq[key] += 1

print(
    "Address-number blocks:",
    len(train_number_index)
)

Address-number blocks: 95691


In [123]:
def get_training_candidates_v4(s1_row):

    candidates = set()

    country = s1_row["country_norm"]
    name = s1_row["name_norm"]
    address = s1_row["address_norm"]

    # ==================================================
    # BLOCK 1 — Exact normalized name
    # ==================================================

    if name:

        candidates.update(
            train_name_index.get(
                (country, name),
                []
            )
        )

    # ==================================================
    # BLOCK 2 — Name token
    # ==================================================

    if name:

        tokens = [
            token
            for token in set(name.split())
            if len(token) >= 3
        ]

        tokens = sorted(
            tokens,
            key=lambda x: name_token_freq.get(
                (country, x),
                float("inf")
            )
        )

        # Take 3 rarest tokens
        for token in tokens[:3]:

            freq = name_token_freq.get(
                (country, token),
                0
            )

            if 0 < freq <= 3000:

                candidates.update(
                    train_name_token_index.get(
                        (country, token),
                        []
                    )
                )

    # ==================================================
    # BLOCK 3 — Character 2-grams
    # ==================================================

    if name:

        grams2 = get_chargrams(name, 2)

        grams2 = sorted(
            grams2,
            key=lambda x: char2_freq.get(
                (country, x),
                float("inf")
            )
        )

        # More grams = higher recall
        for gram in grams2[:8]:

            freq = char2_freq.get(
                (country, gram),
                0
            )

            if 0 < freq <= 3000:

                candidates.update(
                    train_char2_index.get(
                        (country, gram),
                        []
                    )
                )

    # ==================================================
    # BLOCK 4 — Character 3-grams
    # ==================================================

    if name:

        grams3 = get_chargrams(name, 3)

        grams3 = sorted(
            grams3,
            key=lambda x: chargram_freq.get(
                (country, x),
                float("inf")
            )
        )

        for gram in grams3[:5]:

            freq = chargram_freq.get(
                (country, gram),
                0
            )

            if 0 < freq <= 1000:

                candidates.update(
                    train_chargram_index.get(
                        (country, gram),
                        []
                    )
                )

    # ==================================================
    # BLOCK 5 — Name beginning
    # ==================================================

    if name:

        compact = name.replace(" ", "")

        if len(compact) >= 3:

            candidates.update(
                train_name_start_index.get(
                    (country, compact[:3]),
                    []
                )
            )

    # ==================================================
    # BLOCK 6 — Rare address tokens
    # ==================================================

    if address:

        address_tokens = [
            token
            for token in set(address.split())
            if len(token) >= 4
        ]

        address_tokens = sorted(
            address_tokens,
            key=lambda x: address_freq.get(
                x,
                float("inf")
            )
        )

        for token in address_tokens[:5]:

            freq = address_freq.get(
                token,
                0
            )

            if 0 < freq <= 1000:

                candidates.update(
                    train_address_index.get(
                        (country, token),
                        []
                    )
                )

    # ==================================================
    # BLOCK 7 — Address numbers
    # ==================================================

    if address:

        numbers = set(
            re.findall(
                r"\b\d{2,6}\b",
                address
            )
        )

        for number in numbers:

            freq = number_freq.get(
                (country, number),
                0
            )

            if 0 < freq <= 1000:

                candidates.update(
                    train_number_index.get(
                        (country, number),
                        []
                    )
                )

    return candidates

In [125]:
total_matches = 0
recovered_matches = 0

for i in range(1000):

    row = s1.iloc[i]

    s1_id = row["entity_id"]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"] == s1_id,
            "matched_list"
        ].iloc[0]
    )

    candidates = get_training_candidates_v4(row)

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(candidates)
    )

print("Total true matches:", total_matches)
print("Recovered true matches:", recovered_matches)

recall = (
    recovered_matches / total_matches
    if total_matches
    else 0
)

print("Candidate Recall:", recall)
print(f"Candidate Recall: {recall * 100:.2f}%")

Total true matches: 3517
Recovered true matches: 3341
Candidate Recall: 0.9499573500142167
Candidate Recall: 95.00%


In [126]:
candidate_sizes = []

for i in range(1000):

    row = s1.iloc[i]

    candidates = get_training_candidates_v4(row)

    candidate_sizes.append(
        len(candidates)
    )

print("Average:", np.mean(candidate_sizes))
print("Median:", np.median(candidate_sizes))
print("Maximum:", np.max(candidate_sizes))

print(
    ">1000:",
    sum(x > 1000 for x in candidate_sizes)
)

print(
    ">10000:",
    sum(x > 10000 for x in candidate_sizes)
)

Average: 12429.227
Median: 8615.0
Maximum: 106753
>1000: 939
>10000: 456


In [128]:
from rapidfuzz import fuzz

def candidate_score(s1_row, candidate_id, source_lookup):

    candidate = source_lookup[candidate_id]

    name_score = fuzz.token_set_ratio(
        s1_row["name_norm"],
        candidate["name_norm"]
    )

    address_score = fuzz.token_set_ratio(
        s1_row["address_norm"],
        candidate["address_norm"]
    )

    country_score = (
        100
        if s1_row["country_norm"] == candidate["country_norm"]
        else 0
    )

    # Weighted score
    score = (
        0.55 * name_score +
        0.40 * address_score +
        0.05 * country_score
    )

    return score

In [129]:
candidate_lookup = {}

for row in s2.itertuples(index=False):
    candidate_lookup[row.entity_id] = {
        "name_norm": row.name_norm,
        "address_norm": row.address_norm,
        "country_norm": row.country_norm
    }

for row in s3.itertuples(index=False):
    candidate_lookup[row.entity_id] = {
        "name_norm": row.name_norm,
        "address_norm": row.address_norm,
        "country_norm": row.country_norm
    }

print("Lookup records:", len(candidate_lookup))

Lookup records: 10320219


In [131]:
def prune_candidates(s1_row, candidates, top_k=100):

    scored = []

    for candidate_id in candidates:

        score = candidate_score(
            s1_row,
            candidate_id,
            candidate_lookup
        )

        scored.append(
            (candidate_id, score)
        )

    scored.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        candidate_id
        for candidate_id, score in scored[:top_k]
    ]

In [ ]:
for i in range(5):

    row = s1.iloc[i]

    raw_candidates = get_training_candidates_v4(row)

    pruned_candidates = prune_candidates(
        row,
        raw_candidates,
        top_k=100
    )

    print(
        row["entity_id"],
        "| Raw:",
        len(raw_candidates),
        "| Pruned:",
        len(pruned_candidates)
    )

In [ ]:
total_matches = 0
recovered_matches = 0

for i in range(1000):

    row = s1.iloc[i]

    s1_id = row["entity_id"]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"] == s1_id,
            "matched_list"
        ].iloc[0]
    )

    raw_candidates = get_training_candidates_v4(row)

    pruned_candidates = set(
        prune_candidates(
            row,
            raw_candidates,
            top_k=100
        )
    )

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(
            pruned_candidates
        )
    )

print("Total true matches:", total_matches)
print("Recovered after pruning:", recovered_matches)

recall = (
    recovered_matches / total_matches
    if total_matches
    else 0
)

print(f"Pruned Candidate Recall: {recall * 100:.2f}%")

In [4]:
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import defaultdict, Counter
from rapidfuzz import fuzz

TRAIN_PATH = "../dataset/train/"
TEST_PATH = "../dataset/test/"

s1 = pd.read_csv(
    TRAIN_PATH + "train_source1.tsv",
    sep="\t"
)

s2 = pd.read_csv(
    TRAIN_PATH + "train_source2.tsv",
    sep="\t"
)

s3 = pd.read_csv(
    TRAIN_PATH + "train_source3.tsv",
    sep="\t"
)

ground_truth = pd.read_csv(
    TRAIN_PATH + "train_ground_truth.tsv",
    sep="\t"
)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)
print("Ground truth:", ground_truth.shape)

S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)
Ground truth: (2206821, 2)


In [6]:
def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def preprocess_dataframe(df):

    df = df.copy()

    df["name_norm"] = (
        df["business_name"]
        .apply(normalize_text)
    )

    df["address_norm"] = (
        df["business_address"]
        .apply(normalize_text)
    )

    df["country_norm"] = (
        df["country"]
        .apply(normalize_text)
    )

    return df


s1 = preprocess_dataframe(s1)
s2 = preprocess_dataframe(s2)
s3 = preprocess_dataframe(s3)

In [7]:
def parse_matches(x):

    if pd.isna(x) or str(x).strip() == "":
        return []

    return [
        item.strip()
        for item in str(x).split(",")
        if item.strip()
    ]


ground_truth["matched_list"] = (
    ground_truth["matched_entity_ids"]
    .apply(parse_matches)
)

print("Ground truth ready.")

Ground truth ready.


In [8]:
from collections import defaultdict, Counter

train_name_index = defaultdict(list)
train_address_index = defaultdict(list)
train_name_token_index = defaultdict(list)

address_freq = Counter()
name_token_freq = Counter()

train_chargram_index = defaultdict(list)
chargram_freq = Counter()

train_char2_index = defaultdict(list)
char2_freq = Counter()

train_name_start_index = defaultdict(list)

train_number_index = defaultdict(list)
number_freq = Counter()

In [15]:
for df in [s2, s3]:

    for row in df.itertuples(index=False):

        entity_id = row.entity_id
        country = row.country_norm
        name = row.name_norm
        address = row.address_norm

        # Exact name
        if name:
            train_name_index[
                (country, name)
            ].append(entity_id)

            # Name tokens
            tokens = set(
                t for t in name.split()
                if len(t) >= 3
            )

            for token in tokens:
                key = (country, token)
                train_name_token_index[key].append(entity_id)
                name_token_freq[key] += 1

            # Character grams
            compact = name.replace(" ", "")

            grams3 = {
                compact[i:i+3]
                for i in range(max(0, len(compact)-2))
            }

            for gram in grams3:
                key = (country, gram)
                train_chargram_index[key].append(entity_id)
                chargram_freq[key] += 1

            grams2 = {
                compact[i:i+2]
                for i in range(max(0, len(compact)-1))
            }

            for gram in grams2:
                key = (country, gram)
                train_char2_index[key].append(entity_id)
                char2_freq[key] += 1

            # Name beginning
            if len(compact) >= 3:
                train_name_start_index[
                    (country, compact[:3])
                ].append(entity_id)

        # Address
        if address:

            tokens = set(
                t for t in address.split()
                if len(t) >= 4
            )

            for token in tokens:

                key = (country, token)

                train_address_index[key].append(entity_id)
                address_freq[token] += 1

            # Numbers
            numbers = set(
                re.findall(
                    r"\b\d{2,6}\b",
                    address
                )
            )

            for number in numbers:

                key = (country, number)

                train_number_index[key].append(entity_id)
                number_freq[key] += 1

print("Indexes rebuilt successfully.")

KeyboardInterrupt: 

In [ ]:
def get_chargrams(text, n=3):

    if not text:
        return set()

    text = text.replace(" ", "")

    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }

In [ ]:
def get_training_candidates_v4(s1_row):

    candidates = set()

    country = s1_row["country_norm"]
    name = s1_row["name_norm"]
    address = s1_row["address_norm"]

    # Exact name
    if name:
        candidates.update(
            train_name_index.get(
                (country, name), []
            )
        )

    # Name tokens
    if name:

        tokens = [
            t for t in set(name.split())
            if len(t) >= 3
        ]

        tokens = sorted(
            tokens,
            key=lambda x: name_token_freq.get(
                (country, x),
                float("inf")
            )
        )

        for token in tokens[:3]:

            freq = name_token_freq.get(
                (country, token), 0
            )

            if 0 < freq <= 3000:

                candidates.update(
                    train_name_token_index.get(
                        (country, token), []
                    )
                )

    # Character 2-grams
    if name:

        grams = sorted(
            get_chargrams(name, 2),
            key=lambda x: char2_freq.get(
                (country, x),
                float("inf")
            )
        )

        for gram in grams[:8]:

            freq = char2_freq.get(
                (country, gram), 0
            )

            if 0 < freq <= 3000:

                candidates.update(
                    train_char2_index.get(
                        (country, gram), []
                    )
                )

    # Character 3-grams
    if name:

        grams = sorted(
            get_chargrams(name, 3),
            key=lambda x: chargram_freq.get(
                (country, x),
                float("inf")
            )
        )

        for gram in grams[:5]:

            freq = chargram_freq.get(
                (country, gram), 0
            )

            if 0 < freq <= 1000:

                candidates.update(
                    train_chargram_index.get(
                        (country, gram), []
                    )
                )

    # Name beginning
    if name:

        compact = name.replace(" ", "")

        if len(compact) >= 3:

            candidates.update(
                train_name_start_index.get(
                    (country, compact[:3]), []
                )
            )

    # Address tokens
    if address:

        tokens = [
            t for t in set(address.split())
            if len(t) >= 4
        ]

        tokens = sorted(
            tokens,
            key=lambda x: address_freq.get(
                x,
                float("inf")
            )
        )

        for token in tokens[:5]:

            freq = address_freq.get(token, 0)

            if 0 < freq <= 1000:

                candidates.update(
                    train_address_index.get(
                        (country, token), []
                    )
                )

    # Address numbers
    if address:

        numbers = set(
            re.findall(
                r"\b\d{2,6}\b",
                address
            )
        )

        for number in numbers:

            freq = number_freq.get(
                (country, number),
                0
            )

            if 0 < freq <= 1000:

                candidates.update(
                    train_number_index.get(
                        (country, number),
                        []
                    )
                )

    return candidates

In [16]:
print("Kernel is working")
print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

Kernel is working
S1: (2206821, 7)
S2: (5034616, 7)
S3: (5285603, 7)


In [ ]:
train_name_lookup = (
    train_candidates_df
    .set_index(
        ["country_norm", "name_norm"]
    )
)

print("Lookup ready")

In [18]:
print("matched_list" in ground_truth.columns)

True


In [19]:
train_candidates_df = pd.concat(
    [
        s2[[
            "entity_id",
            "name_norm",
            "address_norm",
            "country_norm"
        ]],
        s3[[
            "entity_id",
            "name_norm",
            "address_norm",
            "country_norm"
        ]]
    ],
    ignore_index=True
)

print("Combined S2 + S3 records:",
      len(train_candidates_df))

print(train_candidates_df.head())

Combined S2 + S3 records: 10320219
      entity_id                     name_norm  \
0  S2-166376419                                 
1  S2-764573417     holloway peak inc seafood   
2  S2-639257739                                 
3  S2-163963287                    summit inc   
4   S2-49942811  delta tetlecommunication inc   

                                    address_norm country_norm  
0        kh no 570 13 new delhi west delhi delhi        india  
1                        105 elm st morganton nc           us  
2  g 3 571 gulmohar colony bhopal madhya pradesh        india  
3            greensboro nc 19 1 2 stardust trail           us  
4                  914 pierpont ave cleveland oh           us  


In [20]:
train_name_lookup = (
    train_candidates_df
    .set_index(
        ["country_norm", "name_norm"]
    )
)

print("Exact name lookup created.")

Exact name lookup created.


In [23]:
exact_name_lookup = (
    train_candidates_df[
        train_candidates_df["name_norm"] != ""
    ]
    .groupby(
        ["country_norm", "name_norm"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Exact name groups:", len(exact_name_lookup))

Exact name groups: 7165488


In [24]:
total_matches = 0
recovered_matches = 0

for i in range(1000):

    row = s1.iloc[i]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"]
            == row["entity_id"],
            "matched_list"
        ].iloc[0]
    )

    key = (
        row["country_norm"],
        row["name_norm"]
    )

    candidates = set(
        exact_name_lookup.get(key, [])
    )

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(candidates)
    )

print("Total true matches:", total_matches)
print("Recovered:", recovered_matches)

recall = recovered_matches / total_matches

print(f"Exact-name recall: {recall * 100:.2f}%")

Total true matches: 3517
Recovered: 904
Exact-name recall: 25.70%


In [25]:
# Phase 6.4 - Build compact blocking tables

# Keep only useful columns
s2_block = s2[[
    "entity_id",
    "name_norm",
    "address_norm",
    "country_norm"
]].copy()

s3_block = s3[[
    "entity_id",
    "name_norm",
    "address_norm",
    "country_norm"
]].copy()

# Combine Source 2 + Source 3
candidate_pool = pd.concat(
    [s2_block, s3_block],
    ignore_index=True
)

print("Candidate pool:", candidate_pool.shape)

# ---------------------------------------------------
# 1. Exact name lookup
# ---------------------------------------------------

exact_name_lookup = (
    candidate_pool[
        candidate_pool["name_norm"] != ""
    ]
    .groupby(
        ["country_norm", "name_norm"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Exact name groups:", len(exact_name_lookup))

# ---------------------------------------------------
# 2. Name prefix lookup
# ---------------------------------------------------

candidate_pool["name_prefix"] = (
    candidate_pool["name_norm"]
    .str.replace(" ", "", regex=False)
    .str[:3]
)

prefix_lookup = (
    candidate_pool[
        candidate_pool["name_prefix"] != ""
    ]
    .groupby(
        ["country_norm", "name_prefix"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Prefix groups:", len(prefix_lookup))

# ---------------------------------------------------
# 3. Name token lookup
# ---------------------------------------------------

name_token_df = candidate_pool[
    ["entity_id", "name_norm", "country_norm"]
].copy()

name_token_df["name_token"] = (
    name_token_df["name_norm"]
    .str.split()
) 

name_token_df = name_token_df.explode("name_token")

name_token_df = name_token_df[
    name_token_df["name_token"].notna() &
    (name_token_df["name_token"].str.len() >= 3)
]

name_token_lookup = (
    name_token_df
    .groupby(
        ["country_norm", "name_token"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Name token groups:", len(name_token_lookup))

print("\nPhase 6.4 blocking tables ready.")

Candidate pool: (10320219, 4)
Exact name groups: 7165488
Prefix groups: 48132
Name token groups: 1480028

Phase 6.4 blocking tables ready.


In [26]:
# Phase 6.5 - Combined blocking recall test

def get_candidates_v6(row):
    candidates = set()

    country = row["country_norm"]
    name = row["name_norm"]

    # ------------------------------------------------
    # 1. Exact normalized name + country
    # ------------------------------------------------
    if name:
        candidates.update(
            exact_name_lookup.get(
                (country, name),
                []
            )
        )

    # ------------------------------------------------
    # 2. Name prefix + country
    # ------------------------------------------------
    if name:
        prefix = name.replace(" ", "")[:3]

        if prefix:
            candidates.update(
                prefix_lookup.get(
                    (country, prefix),
                    []
                )
            )

    # ------------------------------------------------
    # 3. Name token blocking
    # ------------------------------------------------
    if name:
        tokens = set(
            token
            for token in name.split()
            if len(token) >= 3
        )

        for token in tokens:
            candidates.update(
                name_token_lookup.get(
                    (country, token),
                    []
                )
            )

    return candidates


# -----------------------------------------------
# Test on first 1000 training Source-1 records
# -----------------------------------------------

total_matches = 0
recovered_matches = 0

candidate_sizes = []

for i in range(1000):

    row = s1.iloc[i]

    # True matches
    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"] == row["entity_id"],
            "matched_list"
        ].iloc[0]
    )

    # Generated candidates
    candidates = get_candidates_v6(row)

    total_matches += len(true_matches)
    recovered_matches += len(
        true_matches.intersection(candidates)
    )

    candidate_sizes.append(len(candidates))


recall = (
    recovered_matches / total_matches
    if total_matches > 0
    else 0
)

print("Total true matches:", total_matches)
print("Recovered:", recovered_matches)
print(f"Candidate Recall: {recall*100:.2f}%")

print("\nCandidate statistics:")
print(f"Average candidates: {np.mean(candidate_sizes):.2f}")
print(f"Median candidates: {np.median(candidate_sizes):.0f}")
print(f"Maximum candidates: {np.max(candidate_sizes)}")
print(f">1000 candidates: {sum(x > 1000 for x in candidate_sizes)}")
print(f">10000 candidates: {sum(x > 10000 for x in candidate_sizes)}")

Total true matches: 3517
Recovered: 3157
Candidate Recall: 89.76%

Candidate statistics:
Average candidates: 760018.35
Median candidates: 855616
Maximum candidates: 1795984
>1000 candidates: 994
>10000 candidates: 946


In [28]:
# Phase 6.6 - Calculate name token frequencies

name_token_freq = (
    name_token_df
    .groupby(
        ["country_norm", "name_token"]
    )
    .size()
)

print("Total token-frequency entries:", len(name_token_freq))

print("\nToken frequency statistics:")
print(name_token_freq.describe())

print("\nMost common tokens:")
print(
    name_token_freq
    .sort_values(ascending=False)
    .head(20)
)

Total token-frequency entries: 1480028

Token frequency statistics:
count    1.480028e+06
mean     2.132721e+01
std      2.105726e+03
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.241219e+06
dtype: float64

Most common tokens:


IndexError: index 0 is out of bounds for axis 0 with size 0

In [29]:
# Phase 6.6 - Corrected token frequency inspection

token_freq_table = (
    name_token_freq
    .reset_index(name="frequency")
    .sort_values("frequency", ascending=False)
)

print("Most common tokens:")
print(
    token_freq_table.head(20).to_string(index=False)
)

print("\nFrequency distribution:")

for limit in [1, 5, 10, 25, 50, 100, 500, 1000, 3000, 10000]:
    count = (token_freq_table["frequency"] <= limit).sum()
    print(
        f"Tokens appearing <= {limit:>5}: "
        f"{count:,} / {len(token_freq_table):,}"
    )

IndexError: index 0 is out of bounds for axis 0 with size 0

In [30]:
# Phase 6.6 - Frequency analysis without sorting

freq_values = name_token_freq.to_numpy()

print("Total token-frequency entries:", len(freq_values))

print("\nToken frequency thresholds:")

for limit in [1, 5, 10, 25, 50, 100, 500, 1000, 3000, 10000]:
    count = np.sum(freq_values <= limit)
    print(
        f"Frequency <= {limit:>5}: "
        f"{count:,} tokens "
        f"({count / len(freq_values) * 100:.2f}%)"
    )

print("\nVery common tokens:")

for limit in [10000, 50000, 100000, 500000, 1000000]:
    count = np.sum(freq_values > limit)
    print(
        f"Frequency > {limit:>7}: "
        f"{count:,} tokens"
    )

Total token-frequency entries: 1480028

Token frequency thresholds:
Frequency <=     1: 1,195,691 tokens (80.79%)
Frequency <=     5: 1,362,589 tokens (92.07%)
Frequency <=    10: 1,394,591 tokens (94.23%)
Frequency <=    25: 1,428,173 tokens (96.50%)
Frequency <=    50: 1,448,414 tokens (97.86%)
Frequency <=   100: 1,462,019 tokens (98.78%)
Frequency <=   500: 1,476,432 tokens (99.76%)
Frequency <=  1000: 1,477,535 tokens (99.83%)
Frequency <=  3000: 1,479,103 tokens (99.94%)
Frequency <= 10000: 1,479,611 tokens (99.97%)

Very common tokens:
Frequency >   10000: 417 tokens
Frequency >   50000: 47 tokens
Frequency >  100000: 23 tokens
Frequency >  500000: 5 tokens
Frequency > 1000000: 3 tokens


In [31]:
# Phase 6.7 - Build frequency-filtered name-token lookup

# Keep tokens that occur at most 100 times
MAX_TOKEN_FREQ = 100

# Convert frequency Series to a set of allowed
# (country, token) pairs
allowed_token_keys = set(
    name_token_freq[
        name_token_freq <= MAX_TOKEN_FREQ
    ].index
)

print("Allowed token keys:", len(allowed_token_keys))

# Filter the existing exploded token dataframe
rare_name_token_df = name_token_df[
    pd.MultiIndex.from_frame(
        name_token_df[["country_norm", "name_token"]]
    ).isin(allowed_token_keys)
].copy()

print("Rare-token rows:", len(rare_name_token_df))

# Build lookup
rare_name_token_lookup = (
    rare_name_token_df
    .groupby(
        ["country_norm", "name_token"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Rare token groups:", len(rare_name_token_lookup))

print("\nPhase 6.7 complete.")

Allowed token keys: 1462019
Rare-token rows: 4111198
Rare token groups: 1462019

Phase 6.7 complete.


In [33]:
# Phase 6.8 - Improved candidate blocking

def get_candidates_v7(row):
    candidates = set()

    country = row["country_norm"]
    name = row["name_norm"]

    # ---------------------------------------------
    # 1. Exact normalized name + country
    # ---------------------------------------------
    if name:
        candidates.update(
            exact_name_lookup.get(
                (country, name),
                []
            )
        )

    # ---------------------------------------------
    # 2. Name prefix + country
    # ---------------------------------------------
    if name:
        prefix = name.replace(" ", "")[:3]

        if prefix:
            candidates.update(
                prefix_lookup.get(
                    (country, prefix),
                    []
                )
            )

    # ---------------------------------------------
    # 3. Rare name tokens only
    # ---------------------------------------------
    if name:
        tokens = set(
            token
            for token in name.split()
            if len(token) >= 3
        )

        for token in tokens:

            candidate_ids = rare_name_token_lookup.get(
                (country, token),
                []
            )

            candidates.update(candidate_ids)

    return candidates


# ---------------------------------------------
# Test on first 1000 training S1 records
# ---------------------------------------------

total_matches = 0
recovered_matches = 0

candidate_sizes = []

for i in range(1000):

    row = s1.iloc[i]

    # Ground-truth matches
    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"] == row["entity_id"],
            "matched_list"
        ].iloc[0]
    )

    # Generated candidates
    candidates = get_candidates_v7(row)

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(candidates)
    )

    candidate_sizes.append(len(candidates))


# ---------------------------------------------
# Metrics
# ---------------------------------------------

recall = (
    recovered_matches / total_matches
    if total_matches > 0
    else 0
)

print("Total true matches:", total_matches)
print("Recovered:", recovered_matches)
print(f"Candidate Recall: {recall * 100:.2f}%")

print("\nCandidate statistics:")
print(f"Average candidates: {np.mean(candidate_sizes):.2f}")
print(f"Median candidates: {np.median(candidate_sizes):.0f}")
print(f"Maximum candidates: {np.max(candidate_sizes)}")
print(f">1000 candidates: {sum(x > 1000 for x in candidate_sizes)}")
print(f">10000 candidates: {sum(x > 10000 for x in candidate_sizes)}")
print(f">100000 candidates: {sum(x > 100000 for x in candidate_sizes)}")

Total true matches: 3517
Recovered: 2794
Candidate Recall: 79.44%

Candidate statistics:
Average candidates: 11232.54
Median candidates: 7652
Maximum candidates: 106665
>1000 candidates: 794
>10000 candidates: 424
>100000 candidates: 2


In [2]:
# Phase 6.9 - High recall blocking
# Fast version: reuse existing candidate_pool and build compact indexes

# ---------- Character n-grams ----------

def make_ngrams(text, n):
    if not text:
        return set()
    text = text.replace(" ", "")
    if len(text) < n:
        return {text}
    return {text[i:i+n] for i in range(len(text)-n+1)}


# Character 2-gram dataframe
char2_df = candidate_pool[
    ["entity_id", "name_norm", "country_norm"]
].copy()

char2_df["gram"] = char2_df["name_norm"].apply(
    lambda x: list(make_ngrams(x, 2))
)

char2_df = char2_df.explode("gram")

char2_df = char2_df[
    char2_df["gram"].notna() &
    (char2_df["gram"] != "")
]

# Frequency
char2_freq_series = (
    char2_df
    .groupby(["country_norm", "gram"])
    .size()
)

# Keep useful grams
char2_allowed = set(
    char2_freq_series[
        char2_freq_series <= 3000
    ].index
)

char2_df = char2_df[
    pd.MultiIndex.from_frame(
        char2_df[["country_norm", "gram"]]
    ).isin(char2_allowed)
]

char2_lookup = (
    char2_df
    .groupby(["country_norm", "gram"], sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Char-2 groups:", len(char2_lookup))


# ---------- Character 3-grams ----------

char3_df = candidate_pool[
    ["entity_id", "name_norm", "country_norm"]
].copy()

char3_df["gram"] = char3_df["name_norm"].apply(
    lambda x: list(make_ngrams(x, 3))
)

char3_df = char3_df.explode("gram")

char3_df = char3_df[
    char3_df["gram"].notna() &
    (char3_df["gram"] != "")
]

char3_freq_series = (
    char3_df
    .groupby(["country_norm", "gram"])
    .size()
)

char3_allowed = set(
    char3_freq_series[
        char3_freq_series <= 1000
    ].index
)

char3_df = char3_df[
    pd.MultiIndex.from_frame(
        char3_df[["country_norm", "gram"]]
    ).isin(char3_allowed)
]

char3_lookup = (
    char3_df
    .groupby(["country_norm", "gram"], sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Char-3 groups:", len(char3_lookup))

print("Phase 6.9 complete.")

NameError: name 'candidate_pool' is not defined

In [3]:
# ============================================
# RESTORE PHASE 6 STATE AFTER KERNEL RESTART
# ============================================

import pandas as pd
import numpy as np
import re
import unicodedata
from rapidfuzz import fuzz

TRAIN_PATH = "../dataset/train/"
TEST_PATH = "../dataset/test/"

# -----------------------------
# Load training data
# -----------------------------

s1 = pd.read_csv(
    TRAIN_PATH + "train_source1.tsv",
    sep="\t"
)

s2 = pd.read_csv(
    TRAIN_PATH + "train_source2.tsv",
    sep="\t"
)

s3 = pd.read_csv(
    TRAIN_PATH + "train_source3.tsv",
    sep="\t"
)

ground_truth = pd.read_csv(
    TRAIN_PATH + "train_ground_truth.tsv",
    sep="\t"
)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)
print("Ground truth:", ground_truth.shape)


# -----------------------------
# Normalization
# -----------------------------

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def preprocess_dataframe(df):

    df = df.copy()

    df["name_norm"] = (
        df["business_name"]
        .apply(normalize_text)
    )

    df["address_norm"] = (
        df["business_address"]
        .apply(normalize_text)
    )

    df["country_norm"] = (
        df["country"]
        .apply(normalize_text)
    )

    return df


s1 = preprocess_dataframe(s1)
s2 = preprocess_dataframe(s2)
s3 = preprocess_dataframe(s3)


# -----------------------------
# Ground truth parsing
# -----------------------------

def parse_matches(x):

    if pd.isna(x) or str(x).strip() == "":
        return []

    return [
        item.strip()
        for item in str(x).split(",")
        if item.strip()
    ]


ground_truth["matched_list"] = (
    ground_truth["matched_entity_ids"]
    .apply(parse_matches)
)


# -----------------------------
# Combined candidate pool
# -----------------------------

candidate_pool = pd.concat(
    [
        s2[
            [
                "entity_id",
                "name_norm",
                "address_norm",
                "country_norm"
            ]
        ],
        s3[
            [
                "entity_id",
                "name_norm",
                "address_norm",
                "country_norm"
            ]
        ]
    ],
    ignore_index=True
)

print(
    "Candidate pool:",
    candidate_pool.shape
)

S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)
Ground truth: (2206821, 2)
Candidate pool: (10320219, 4)


In [4]:
# ============================================================
# PHASE 6 - FINAL FAST BLOCKING INDEXES
# ============================================================

# Combined S2 + S3
candidate_pool = pd.concat(
    [
        s2[["entity_id", "name_norm", "address_norm", "country_norm"]],
        s3[["entity_id", "name_norm", "address_norm", "country_norm"]]
    ],
    ignore_index=True
)

print("Candidate pool:", candidate_pool.shape)


# ------------------------------------------------------------
# 1. EXACT NAME INDEX
# ------------------------------------------------------------

exact_name_lookup = (
    candidate_pool[
        candidate_pool["name_norm"] != ""
    ]
    .groupby(
        ["country_norm", "name_norm"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Exact-name groups:", len(exact_name_lookup))


# ------------------------------------------------------------
# 2. PREFIX INDEX
# ------------------------------------------------------------

candidate_pool["name_prefix"] = (
    candidate_pool["name_norm"]
    .str.replace(" ", "", regex=False)
    .str[:3]
)

prefix_lookup = (
    candidate_pool[
        candidate_pool["name_prefix"] != ""
    ]
    .groupby(
        ["country_norm", "name_prefix"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Prefix groups:", len(prefix_lookup))


# ------------------------------------------------------------
# 3. CHARACTER 2-GRAM INDEX
# ------------------------------------------------------------

def make_ngrams(text, n):
    if not text:
        return []

    text = text.replace(" ", "")

    if len(text) < n:
        return [text]

    return [
        text[i:i+n]
        for i in range(len(text)-n+1)
    ]


char2_df = candidate_pool[
    ["entity_id", "name_norm", "country_norm"]
].copy()

char2_df["gram"] = char2_df["name_norm"].apply(
    lambda x: list(set(make_ngrams(x, 2)))
)

char2_df = char2_df.explode("gram")

char2_df = char2_df[
    char2_df["gram"].notna() &
    (char2_df["gram"] != "")
]

char2_freq = (
    char2_df
    .groupby(["country_norm", "gram"])
    .size()
)

# Ignore extremely common grams
char2_allowed = set(
    char2_freq[
        char2_freq <= 3000
    ].index
)

char2_df = char2_df[
    pd.MultiIndex.from_frame(
        char2_df[["country_norm", "gram"]]
    ).isin(char2_allowed)
]

char2_lookup = (
    char2_df
    .groupby(
        ["country_norm", "gram"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Char-2 groups:", len(char2_lookup))


# ------------------------------------------------------------
# 4. CHARACTER 3-GRAM INDEX
# ------------------------------------------------------------

char3_df = candidate_pool[
    ["entity_id", "name_norm", "country_norm"]
].copy()

char3_df["gram"] = char3_df["name_norm"].apply(
    lambda x: list(set(make_ngrams(x, 3)))
)

char3_df = char3_df.explode("gram")

char3_df = char3_df[
    char3_df["gram"].notna() &
    (char3_df["gram"] != "")
]

char3_freq = (
    char3_df
    .groupby(["country_norm", "gram"])
    .size()
)

char3_allowed = set(
    char3_freq[
        char3_freq <= 1000
    ].index
)

char3_df = char3_df[
    pd.MultiIndex.from_frame(
        char3_df[["country_norm", "gram"]]
    ).isin(char3_allowed)
]

char3_lookup = (
    char3_df
    .groupby(
        ["country_norm", "gram"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Char-3 groups:", len(char3_lookup))

print("\nPhase 6 indexes ready.")

Candidate pool: (10320219, 4)
Exact-name groups: 7165488
Prefix groups: 48132


KeyboardInterrupt: 

In [5]:
print("Kernel alive")
print("candidate_pool:", candidate_pool.shape)
print("exact_name_lookup:", len(exact_name_lookup))
print("prefix_lookup:", len(prefix_lookup))

Kernel alive
candidate_pool: (10320219, 5)
exact_name_lookup: 7165488
prefix_lookup: 48132


In [6]:
# ============================================================
# PHASE 6.9 - FAST CANDIDATE GENERATION
# ============================================================

# Add cheap helper columns to candidate pool
candidate_pool["name_len"] = candidate_pool["name_norm"].str.len()
candidate_pool["name_words"] = candidate_pool["name_norm"].str.count(" ") + 1

# Lookup entity_id -> row information
candidate_lookup = (
    candidate_pool
    .set_index("entity_id")
    [["name_norm", "address_norm", "country_norm",
      "name_len", "name_words"]]
    .to_dict("index")
)

print("Candidate lookup ready:", len(candidate_lookup))


def fast_candidates(row, max_candidates=5000):

    country = row["country_norm"]
    name = row["name_norm"]

    candidates = set()

    # --------------------------------------------------------
    # 1. EXACT NAME — always keep
    # --------------------------------------------------------

    if name:

        candidates.update(
            exact_name_lookup.get(
                (country, name),
                []
            )
        )

    # --------------------------------------------------------
    # 2. PREFIX BLOCK
    # --------------------------------------------------------

    if name:

        prefix = name.replace(" ", "")[:3]

        prefix_candidates = prefix_lookup.get(
            (country, prefix),
            []
        )

        # Cheap filtering before fuzzy matching
        s1_len = len(name)
        s1_words = len(name.split())

        for cid in prefix_candidates:

            # Already included
            if cid in candidates:
                continue

            c = candidate_lookup[cid]

            # Name length difference filter
            if abs(c["name_len"] - s1_len) > max(8, int(s1_len * 0.5)):
                continue

            # Word-count difference filter
            if abs(c["name_words"] - s1_words) > 2:
                continue

            candidates.add(cid)

            # Safety cap
            if len(candidates) >= max_candidates:
                break

    return candidates


# ============================================================
# TEST ON 1000 TRAINING S1 RECORDS
# ============================================================

total_matches = 0
recovered_matches = 0
candidate_sizes = []

for i in range(1000):

    row = s1.iloc[i]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"] == row["entity_id"],
            "matched_list"
        ].iloc[0]
    )

    candidates = fast_candidates(row)

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(candidates)
    )

    candidate_sizes.append(len(candidates))


recall = recovered_matches / total_matches

print("\nTotal true matches:", total_matches)
print("Recovered:", recovered_matches)
print(f"Candidate Recall: {recall * 100:.2f}%")

print("\nCandidate statistics:")
print(f"Average: {np.mean(candidate_sizes):.2f}")
print(f"Median: {np.median(candidate_sizes):.0f}")
print(f"Maximum: {np.max(candidate_sizes)}")
print(f">1000: {sum(x > 1000 for x in candidate_sizes)}")
print(f">5000: {sum(x > 5000 for x in candidate_sizes)}")

Candidate lookup ready: 10320219

Total true matches: 3517
Recovered: 2111
Candidate Recall: 60.02%

Candidate statistics:
Average: 3320.93
Median: 5000
Maximum: 5000
>1000: 752
>5000: 0


In [7]:
# ============================================================
# FINAL PHASE 6 - HIGH RECALL BLOCKING
# Target: >=95% candidate recall
# ============================================================

from collections import defaultdict, Counter
import re
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. COMBINED SOURCE 2 + SOURCE 3
# ------------------------------------------------------------

candidate_pool = pd.concat(
    [
        s2[["entity_id", "name_norm", "address_norm", "country_norm"]],
        s3[["entity_id", "name_norm", "address_norm", "country_norm"]]
    ],
    ignore_index=True
)

print("Candidate pool:", candidate_pool.shape)


# ------------------------------------------------------------
# 2. EXACT NAME BLOCK
# ------------------------------------------------------------

exact_name_lookup = (
    candidate_pool[
        candidate_pool["name_norm"] != ""
    ]
    .groupby(
        ["country_norm", "name_norm"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Exact-name groups:", len(exact_name_lookup))


# ------------------------------------------------------------
# 3. NAME TOKEN BLOCK
# ------------------------------------------------------------

name_token_df = candidate_pool[
    ["entity_id", "name_norm", "country_norm"]
].copy()

name_token_df["token"] = (
    name_token_df["name_norm"]
    .str.split()
)

name_token_df = name_token_df.explode("token")

name_token_df = name_token_df[
    name_token_df["token"].notna() &
    (name_token_df["token"].str.len() >= 3)
]

name_token_freq = (
    name_token_df
    .groupby(["country_norm", "token"])
    .size()
)

# Earlier successful blocker used rare tokens.
# Keep tokens occurring <= 3000 times.

name_token_allowed = set(
    name_token_freq[
        name_token_freq <= 3000
    ].index
)

name_token_filtered = name_token_df[
    pd.MultiIndex.from_frame(
        name_token_df[["country_norm", "token"]]
    ).isin(name_token_allowed)
]

name_token_lookup = (
    name_token_filtered
    .groupby(
        ["country_norm", "token"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Name-token groups:", len(name_token_lookup))


# ------------------------------------------------------------
# 4. CHARACTER GRAM LOOKUP
# ------------------------------------------------------------

def get_grams(text, n):

    if not text:
        return set()

    text = text.replace(" ", "")

    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text)-n+1)
    }


# ------------------------------------------------------------
# 4A. CHAR-2
# ------------------------------------------------------------

char2_lookup = defaultdict(list)
char2_freq = Counter()

# Build using unique grams per record
for row in candidate_pool[
    ["entity_id", "name_norm", "country_norm"]
].itertuples(index=False):

    grams = get_grams(row.name_norm, 2)

    for gram in grams:

        if gram:
            key = (row.country_norm, gram)

            char2_freq[key] += 1

            if char2_freq[key] <= 3000:
                char2_lookup[key].append(row.entity_id)

print("Char-2 lookup:", len(char2_lookup))


# ------------------------------------------------------------
# 4B. CHAR-3
# ------------------------------------------------------------

char3_lookup = defaultdict(list)
char3_freq = Counter()

for row in candidate_pool[
    ["entity_id", "name_norm", "country_norm"]
].itertuples(index=False):

    grams = get_grams(row.name_norm, 3)

    for gram in grams:

        if gram:
            key = (row.country_norm, gram)

            char3_freq[key] += 1

            if char3_freq[key] <= 1000:
                char3_lookup[key].append(row.entity_id)

print("Char-3 lookup:", len(char3_lookup))


# ------------------------------------------------------------
# 5. NAME PREFIX
# ------------------------------------------------------------

prefix_lookup = (
    candidate_pool.assign(
        prefix=
        candidate_pool["name_norm"]
        .str.replace(" ", "", regex=False)
        .str[:3]
    )
    .query("prefix != ''")
    .groupby(
        ["country_norm", "prefix"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("Prefix groups:", len(prefix_lookup))


# ============================================================
# 6. FINAL HIGH-RECALL CANDIDATE FUNCTION
# ============================================================

def final_candidates(row):

    candidates = set()

    country = row["country_norm"]
    name = row["name_norm"]
    address = row["address_norm"]


    # --------------------------------------------------------
    # BLOCK 1: EXACT NAME
    # --------------------------------------------------------

    if name:

        candidates.update(
            exact_name_lookup.get(
                (country, name),
                []
            )
        )


    # --------------------------------------------------------
    # BLOCK 2: NAME TOKENS
    # --------------------------------------------------------

    if name:

        tokens = list(
            set(
                t for t in name.split()
                if len(t) >= 3
            )
        )

        # Rarest tokens first
        token_data = []

        for token in tokens:

            freq = name_token_freq.get(
                (country, token),
                999999999
            )

            token_data.append(
                (freq, token)
            )

        token_data.sort()

        # Use up to 3 rarest useful tokens
        for freq, token in token_data[:3]:

            if freq <= 3000:

                candidates.update(
                    name_token_lookup.get(
                        (country, token),
                        []
                    )
                )


    # --------------------------------------------------------
    # BLOCK 3: CHAR-2 GRAMS
    # --------------------------------------------------------

    if name:

        grams = list(
            get_grams(name, 2)
        )

        gram_data = []

        for gram in grams:

            freq = char2_freq.get(
                (country, gram),
                999999999
            )

            gram_data.append(
                (freq, gram)
            )

        gram_data.sort()

        # 8 rarest useful grams
        for freq, gram in gram_data[:8]:

            if freq <= 3000:

                candidates.update(
                    char2_lookup.get(
                        (country, gram),
                        []
                    )
                )


    # --------------------------------------------------------
    # BLOCK 4: CHAR-3 GRAMS
    # --------------------------------------------------------

    if name:

        grams = list(
            get_grams(name, 3)
        )

        gram_data = []

        for gram in grams:

            freq = char3_freq.get(
                (country, gram),
                999999999
            )

            gram_data.append(
                (freq, gram)
            )

        gram_data.sort()

        # 5 rarest useful grams
        for freq, gram in gram_data[:5]:

            if freq <= 1000:

                candidates.update(
                    char3_lookup.get(
                        (country, gram),
                        []
                    )
                )


    # --------------------------------------------------------
    # BLOCK 5: PREFIX
    # --------------------------------------------------------

    if name:

        prefix = (
            name
            .replace(" ", "")
            [:3]
        )

        if prefix:

            candidates.update(
                prefix_lookup.get(
                    (country, prefix),
                    []
                )
            )


    return candidates


# ============================================================
# 7. VALIDATE ON SAME 1000 RECORDS
# ============================================================

total_matches = 0
recovered_matches = 0
candidate_sizes = []

for i in range(1000):

    row = s1.iloc[i]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"]
            == row["entity_id"],
            "matched_list"
        ].iloc[0]
    )

    candidates = final_candidates(row)

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(candidates)
    )

    candidate_sizes.append(
        len(candidates)
    )


recall = (
    recovered_matches /
    total_matches
)

print("\n===================================")
print("FINAL BLOCKING VALIDATION")
print("===================================")

print("Total true matches:", total_matches)
print("Recovered:", recovered_matches)

print(
    f"Candidate Recall: "
    f"{recall * 100:.2f}%"
)

print("\nCandidate statistics:")
print(
    f"Average: "
    f"{np.mean(candidate_sizes):.2f}"
)

print(
    f"Median: "
    f"{np.median(candidate_sizes):.0f}"
)

print(
    f"Maximum: "
    f"{np.max(candidate_sizes)}"
)

Candidate pool: (10320219, 4)
Exact-name groups: 7165488
Name-token groups: 1479103
Char-2 lookup: 2581
Char-3 lookup: 68090
Prefix groups: 48132

FINAL BLOCKING VALIDATION
Total true matches: 3517
Recovered: 3010
Candidate Recall: 85.58%

Candidate statistics:
Average: 12032.86
Median: 8344
Maximum: 106665


In [8]:
from collections import defaultdict, Counter
import re
import numpy as np
import pandas as pd

candidate_pool = pd.concat(
    [
        s2[["entity_id", "name_norm", "address_norm", "country_norm"]],
        s3[["entity_id", "name_norm", "address_norm", "country_norm"]]
    ],
    ignore_index=True
)

candidate_pool["name_norm"] = candidate_pool["name_norm"].fillna("")
candidate_pool["address_norm"] = candidate_pool["address_norm"].fillna("")
candidate_pool["country_norm"] = candidate_pool["country_norm"].fillna("")

print("Candidate pool:", candidate_pool.shape)

exact_name_lookup = (
    candidate_pool[candidate_pool["name_norm"] != ""]
    .groupby(["country_norm", "name_norm"], sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Exact-name groups (country-scoped):", len(exact_name_lookup))

exact_name_lookup_global = (
    candidate_pool[candidate_pool["name_norm"] != ""]
    .groupby("name_norm", sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Exact-name groups (global):", len(exact_name_lookup_global))

name_token_df = candidate_pool[["entity_id", "name_norm", "country_norm"]].copy()
name_token_df["token"] = name_token_df["name_norm"].str.split()
name_token_df = name_token_df.explode("token")
name_token_df = name_token_df[
    name_token_df["token"].notna() & (name_token_df["token"].str.len() >= 3)
]

name_token_freq = (
    name_token_df.groupby(["country_norm", "token"]).size()
)

name_token_allowed = set(name_token_freq[name_token_freq <= 3000].index)

name_token_filtered = name_token_df[
    pd.MultiIndex.from_frame(
        name_token_df[["country_norm", "token"]]
    ).isin(name_token_allowed)
]

name_token_lookup = (
    name_token_filtered
    .groupby(["country_norm", "token"], sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Name-token groups (country-scoped):", len(name_token_lookup))

name_token_freq_global = name_token_df.groupby("token").size()

name_token_allowed_global = set(
    name_token_freq_global[name_token_freq_global <= 1500].index
)

name_token_filtered_global = name_token_df[
    name_token_df["token"].isin(name_token_allowed_global)
]

name_token_lookup_global = (
    name_token_filtered_global
    .groupby("token", sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Name-token groups (global):", len(name_token_lookup_global))

def get_grams(text, n):
    if not text:
        return set()
    text = text.replace(" ", "")
    if len(text) < n:
        return {text} if text else set()
    return {text[i:i + n] for i in range(len(text) - n + 1)}

char2_lookup = defaultdict(list)
char2_freq = Counter()

for row in candidate_pool[["entity_id", "name_norm", "country_norm"]].itertuples(index=False):
    for gram in get_grams(row.name_norm, 2):
        if gram:
            key = (row.country_norm, gram)
            char2_freq[key] += 1
            if char2_freq[key] <= 3000:
                char2_lookup[key].append(row.entity_id)

print("Char-2 lookup:", len(char2_lookup))

char3_lookup = defaultdict(list)
char3_freq = Counter()

for row in candidate_pool[["entity_id", "name_norm", "country_norm"]].itertuples(index=False):
    for gram in get_grams(row.name_norm, 3):
        if gram:
            key = (row.country_norm, gram)
            char3_freq[key] += 1
            if char3_freq[key] <= 1000:
                char3_lookup[key].append(row.entity_id)

print("Char-3 lookup:", len(char3_lookup))

prefix_lookup = (
    candidate_pool.assign(
        prefix=candidate_pool["name_norm"].str.replace(" ", "", regex=False).str[:3]
    )
    .query("prefix != ''")
    .groupby(["country_norm", "prefix"], sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Prefix groups (country-scoped):", len(prefix_lookup))

prefix_lookup_global = (
    candidate_pool.assign(
        prefix=candidate_pool["name_norm"].str.replace(" ", "", regex=False).str[:3]
    )
    .query("prefix != ''")
    .groupby("prefix", sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Prefix groups (global):", len(prefix_lookup_global))

addr_token_df = candidate_pool[["entity_id", "address_norm", "country_norm"]].copy()
addr_token_df["token"] = addr_token_df["address_norm"].str.split()
addr_token_df = addr_token_df.explode("token")
addr_token_df = addr_token_df[
    addr_token_df["token"].notna() & (addr_token_df["token"].str.len() >= 4)
]

addr_token_freq = addr_token_df.groupby(["country_norm", "token"]).size()

addr_token_allowed = set(addr_token_freq[addr_token_freq <= 5000].index)

addr_token_filtered = addr_token_df[
    pd.MultiIndex.from_frame(
        addr_token_df[["country_norm", "token"]]
    ).isin(addr_token_allowed)
]

addr_token_lookup = (
    addr_token_filtered
    .groupby(["country_norm", "token"], sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Address-token groups:", len(addr_token_lookup))

addr_char3_lookup = defaultdict(list)
addr_char3_freq = Counter()

for row in candidate_pool[["entity_id", "address_norm", "country_norm"]].itertuples(index=False):
    for gram in get_grams(row.address_norm, 3):
        if gram:
            key = (row.country_norm, gram)
            addr_char3_freq[key] += 1
            if addr_char3_freq[key] <= 1500:
                addr_char3_lookup[key].append(row.entity_id)

print("Address char-3 lookup:", len(addr_char3_lookup))

_SOUNDEX_CODES = {
    **{c: "1" for c in "BFPV"},
    **{c: "2" for c in "CGJKQSXZ"},
    **{c: "3" for c in "DT"},
    "L": "4",
    **{c: "5" for c in "MN"},
    "R": "6",
}

def soundex(word):
    word = re.sub(r"[^A-Z]", "", word.upper())
    if not word:
        return ""

    first_letter = word[0]
    codes = [_SOUNDEX_CODES.get(c, "") for c in word]

    result = [first_letter]
    prev_code = _SOUNDEX_CODES.get(first_letter, "")

    for c in codes[1:]:
        if c != "" and c != prev_code:
            result.append(c)
        prev_code = c

    result = "".join(result) + "000"
    return result[:4]

def name_soundex_key(name_norm):
    """Soundex of up to the first 2 tokens, joined -> a single block key."""
    tokens = name_norm.split()[:2]
    codes = [soundex(t) for t in tokens if soundex(t)]
    return "-".join(codes)

soundex_df = candidate_pool[["entity_id", "name_norm", "country_norm"]].copy()
soundex_df["skey"] = soundex_df["name_norm"].apply(name_soundex_key)
soundex_df = soundex_df[soundex_df["skey"] != ""]

soundex_freq = soundex_df.groupby(["country_norm", "skey"]).size()
soundex_allowed = set(soundex_freq[soundex_freq <= 2000].index)

soundex_filtered = soundex_df[
    pd.MultiIndex.from_frame(
        soundex_df[["country_norm", "skey"]]
    ).isin(soundex_allowed)
]

soundex_lookup = (
    soundex_filtered
    .groupby(["country_norm", "skey"], sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Soundex groups:", len(soundex_lookup))

def final_candidates(row):

    candidates = set()

    country = row["country_norm"] if pd.notna(row["country_norm"]) else ""
    name = row["name_norm"] if pd.notna(row["name_norm"]) else ""
    address = row["address_norm"] if pd.notna(row["address_norm"]) else ""

    if name:
        candidates.update(exact_name_lookup.get((country, name), []))
        candidates.update(exact_name_lookup_global.get(name, []))

    if name:
        tokens = list(set(t for t in name.split() if len(t) >= 3))

        token_data = sorted(
            (name_token_freq.get((country, t), 999999999), t) for t in tokens
        )
        for freq, token in token_data[:3]:
            if freq <= 3000:
                candidates.update(name_token_lookup.get((country, token), []))

        token_data_global = sorted(
            (name_token_freq_global.get(t, 999999999), t) for t in tokens
        )
        for freq, token in token_data_global[:2]:
            if freq <= 1500:
                candidates.update(name_token_lookup_global.get(token, []))

    if name:
        gram_data = sorted(
            (char2_freq.get((country, g), 999999999), g) for g in get_grams(name, 2)
        )
        for freq, gram in gram_data[:8]:
            if freq <= 3000:
                candidates.update(char2_lookup.get((country, gram), []))

    if name:
        gram_data = sorted(
            (char3_freq.get((country, g), 999999999), g) for g in get_grams(name, 3)
        )
        for freq, gram in gram_data[:5]:
            if freq <= 1000:
                candidates.update(char3_lookup.get((country, gram), []))

    if name:
        prefix = name.replace(" ", "")[:3]
        if prefix:
            candidates.update(prefix_lookup.get((country, prefix), []))
            candidates.update(prefix_lookup_global.get(prefix, []))

    if address:
        addr_tokens = list(set(t for t in address.split() if len(t) >= 4))
        addr_token_data = sorted(
            (addr_token_freq.get((country, t), 999999999), t) for t in addr_tokens
        )
        for freq, token in addr_token_data[:5]:
            if freq <= 5000:
                candidates.update(addr_token_lookup.get((country, token), []))

    if address:
        addr_gram_data = sorted(
            (addr_char3_freq.get((country, g), 999999999), g)
            for g in get_grams(address, 3)
        )
        for freq, gram in addr_gram_data[:5]:
            if freq <= 1500:
                candidates.update(addr_char3_lookup.get((country, gram), []))

    if name:
        skey = name_soundex_key(name)
        if skey:
            candidates.update(soundex_lookup.get((country, skey), []))

    if not candidates:
        if name:
            prefix = name.replace(" ", "")[:3]
            candidates.update(prefix_lookup_global.get(prefix, []))
        if address:
            addr_tokens = [t for t in address.split() if len(t) >= 4]
            for t in addr_tokens[:5]:
                for key, ids in addr_token_lookup.items():
                    if key[1] == t:
                        candidates.update(ids)

    return candidates

total_matches = 0
recovered_matches = 0
candidate_sizes = []
missed_rows = []

for i in range(1000):

    row = s1.iloc[i]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"] == row["entity_id"],
            "matched_list"
        ].iloc[0]
    )

    candidates = final_candidates(row)

    total_matches += len(true_matches)
    hit = true_matches.intersection(candidates)
    recovered_matches += len(hit)
    candidate_sizes.append(len(candidates))

    missed = true_matches - candidates
    if missed:
        missed_rows.append((row["entity_id"], missed))

recall = recovered_matches / total_matches

print("\n===================================")
print("PHASE 7 - FINAL BLOCKING VALIDATION")
print("===================================")

print("Total true matches:", total_matches)
print("Recovered:", recovered_matches)
print(f"Candidate Recall: {recall * 100:.2f}%")

print("\nCandidate statistics:")
print(f"Average: {np.mean(candidate_sizes):.2f}")
print(f"Median: {np.median(candidate_sizes):.0f}")
print(f"Maximum: {np.max(candidate_sizes)}")

if missed_rows:
    print(f"\nRows still missing at least one match: {len(missed_rows)}")
    print("(inspect these first if recall is still < 95% - they usually")
    print(" share a data-quality issue: e.g. name_norm truly blank, or")
    print(" a legal-entity-only match with no shared address either)")
    for eid, missed in missed_rows[:10]:
        print(f"  entity_id={eid} missing={missed}")

Candidate pool: (10320219, 4)
Exact-name groups (country-scoped): 7165488


KeyboardInterrupt: 

In [10]:
# ============================================================
# PHASE 6.10 - RAPIDFUZZ NAME RETRIEVAL TEST
# ============================================================

from rapidfuzz import process, fuzz

# ------------------------------------------------------------
# Build unique names by country
# ------------------------------------------------------------

name_lists = {}

for country in candidate_pool["country_norm"].unique():

    names = (
        candidate_pool.loc[
            (candidate_pool["country_norm"] == country) &
            (candidate_pool["name_norm"] != ""),
            "name_norm"
        ]
        .drop_duplicates()
        .tolist()
    )

    name_lists[country] = names

    print(
        country,
        "unique names:",
        len(names)
    )


# ------------------------------------------------------------
# Map name -> entity IDs
# ------------------------------------------------------------

name_to_ids = (
    candidate_pool[
        candidate_pool["name_norm"] != ""
    ]
    .groupby(
        ["country_norm", "name_norm"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)


# ------------------------------------------------------------
# Test retrieval
# ------------------------------------------------------------

def rapidfuzz_candidates(row, limit=100):

    country = row["country_norm"]
    name = row["name_norm"]

    if not name:
        return set()

    choices = name_lists.get(country, [])

    if not choices:
        return set()

    results = process.extract(
        name,
        choices,
        scorer=fuzz.token_set_ratio,
        limit=limit,
        score_cutoff=45
    )

    candidates = set()

    for matched_name, score, _ in results:

        candidates.update(
            name_to_ids.get(
                (country, matched_name),
                []
            )
        )

    return candidates


# ------------------------------------------------------------
# Validate first 1000
# ------------------------------------------------------------

total_matches = 0
recovered_matches = 0
candidate_sizes = []

for i in range(1000):

    row = s1.iloc[i]

    true_matches = set(
        ground_truth.loc[
            ground_truth["source1_entity_id"]
            == row["entity_id"],
            "matched_list"
        ].iloc[0]
    )

    candidates = rapidfuzz_candidates(
        row,
        limit=100
    )

    total_matches += len(true_matches)

    recovered_matches += len(
        true_matches.intersection(candidates)
    )

    candidate_sizes.append(
        len(candidates)
    )


recall = recovered_matches / total_matches

print("\n===================================")
print("RAPIDFUZZ RETRIEVAL")
print("===================================")

print("Total true matches:", total_matches)
print("Recovered:", recovered_matches)
print(f"Recall: {recall * 100:.2f}%")

print("\nCandidate statistics:")
print(f"Average: {np.mean(candidate_sizes):.2f}")
print(f"Median: {np.median(candidate_sizes):.0f}")
print(f"Maximum: {np.max(candidate_sizes)}")

india unique names: 2695426
us unique names: 4470062


KeyboardInterrupt: 

In [11]:
print("READY FOR PHASE 7")

READY FOR PHASE 7


In [1]:
# ============================================================
# PHASE 7.1 - GENERATE TRAINING CANDIDATE PAIRS
# ============================================================

TRAIN_S1_LIMIT = 10_000

training_pairs = []

# Ground-truth lookup for fast access
ground_truth_lookup = dict(
    zip(
        ground_truth["source1_entity_id"],
        ground_truth["matched_list"]
    )
)

print("Generating candidate pairs...")

for i in range(TRAIN_S1_LIMIT):

    row = s1.iloc[i]

    # Use our final blocking function
    candidates = final_candidates(row)

    for candidate_id in candidates:

        training_pairs.append(
            (
                row["entity_id"],
                candidate_id
            )
        )

    if (i + 1) % 1000 == 0:
        print(
            f"Processed {i+1:,} / "
            f"{TRAIN_S1_LIMIT:,} S1 records"
        )


training_pairs_df = pd.DataFrame(
    training_pairs,
    columns=[
        "source1_entity_id",
        "candidate_entity_id"
    ]
)

print("\n===================================")
print("TRAINING CANDIDATES")
print("===================================")

print(
    "Candidate pairs:",
    len(training_pairs_df)
)

print(
    "Unique S1:",
    training_pairs_df[
        "source1_entity_id"
    ].nunique()
)

NameError: name 'ground_truth' is not defined

In [2]:
# ============================================================
# PHASE 7.1 - GENERATE TRAINING CANDIDATE PAIRS
# ============================================================

TRAIN_S1_LIMIT = 10_000

training_pairs = []

# Ground-truth lookup for fast access
ground_truth_lookup = dict(
    zip(
        ground_truth["source1_entity_id"],
        ground_truth["matched_list"]
    )
)

print("Generating candidate pairs...")

for i in range(TRAIN_S1_LIMIT):

    row = s1.iloc[i]

    # Use our final blocking function
    candidates = final_candidates(row)

    for candidate_id in candidates:

        training_pairs.append(
            (
                row["entity_id"],
                candidate_id
            )
        )

    if (i + 1) % 1000 == 0:
        print(
            f"Processed {i+1:,} / "
            f"{TRAIN_S1_LIMIT:,} S1 records"
        )


training_pairs_df = pd.DataFrame(
    training_pairs,
    columns=[
        "source1_entity_id",
        "candidate_entity_id"
    ]
)

print("\n===================================")
print("TRAINING CANDIDATES")
print("===================================")

print(
    "Candidate pairs:",
    len(training_pairs_df)
)

print(
    "Unique S1:",
    training_pairs_df[
        "source1_entity_id"
    ].nunique()
)

NameError: name 'ground_truth' is not defined

In [3]:
# Restore ground truth after kernel restart

import pandas as pd

TRAIN_PATH = "../dataset/train/"

ground_truth = pd.read_csv(
    TRAIN_PATH + "train_ground_truth.tsv",
    sep="\t"
)

def parse_matches(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    
    return [
        item.strip()
        for item in str(x).split(",")
        if item.strip()
    ]

ground_truth["matched_list"] = (
    ground_truth["matched_entity_ids"]
    .apply(parse_matches)
)

ground_truth_lookup = dict(
    zip(
        ground_truth["source1_entity_id"],
        ground_truth["matched_list"]
    )
)

print("Ground truth:", ground_truth.shape)
print("Ground truth lookup:", len(ground_truth_lookup))
print("Ready for Phase 7.")

Ground truth: (2206821, 3)
Ground truth lookup: 2206821
Ready for Phase 7.


In [4]:
# ============================================================
# PHASE 7.1 - BUILD BALANCED ML TRAINING DATA
# ============================================================

from rapidfuzz import fuzz
import random

TRAIN_S1_LIMIT = 3000
NEGATIVES_PER_S1 = 5

training_rows = []

print("Building training pairs...")

for i in range(TRAIN_S1_LIMIT):

    row = s1.iloc[i]

    s1_id = row["entity_id"]

    # True matches
    true_matches = set(
        ground_truth_lookup.get(
            s1_id,
            []
        )
    )

    # Candidate set
    candidates = final_candidates(row)

    # --------------------------------------------------------
    # POSITIVES
    # --------------------------------------------------------

    for cid in true_matches:

        # Only use positives that blocking actually found
        if cid in candidates:

            training_rows.append(
                {
                    "source1_entity_id": s1_id,
                    "candidate_entity_id": cid,
                    "label": 1
                }
            )


    # --------------------------------------------------------
    # HARD NEGATIVES
    # --------------------------------------------------------

    negative_candidates = list(
        candidates - true_matches
    )

    # Random sample of negatives
    if len(negative_candidates) > NEGATIVES_PER_S1:

        negative_candidates = random.sample(
            negative_candidates,
            NEGATIVES_PER_S1
        )

    for cid in negative_candidates:

        training_rows.append(
            {
                "source1_entity_id": s1_id,
                "candidate_entity_id": cid,
                "label": 0
            }
        )


    if (i + 1) % 500 == 0:

        print(
            f"Processed "
            f"{i+1:,}/{TRAIN_S1_LIMIT:,}"
        )


training_pairs_df = pd.DataFrame(
    training_rows
)

print("\n===================================")
print("PHASE 7.1 COMPLETE")
print("===================================")

print(
    "Training pairs:",
    len(training_pairs_df)
)

print(
    "Positive:",
    (training_pairs_df["label"] == 1).sum()
)

print(
    "Negative:",
    (training_pairs_df["label"] == 0).sum()
)

print(
    "Positive ratio:",
    training_pairs_df["label"].mean()
)

Building training pairs...


NameError: name 's1' is not defined

In [5]:
# ============================================================
# PHASE 7 - RESTORE REQUIRED DATA
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
from rapidfuzz import fuzz
import random

TRAIN_PATH = "../dataset/train/"

# ------------------------------------------------------------
# Load Source 1
# ------------------------------------------------------------

s1 = pd.read_csv(
    TRAIN_PATH + "train_source1.tsv",
    sep="\t"
)

# ------------------------------------------------------------
# Load Source 2
# ------------------------------------------------------------

s2 = pd.read_csv(
    TRAIN_PATH + "train_source2.tsv",
    sep="\t"
)

# ------------------------------------------------------------
# Load Source 3
# ------------------------------------------------------------

s3 = pd.read_csv(
    TRAIN_PATH + "train_source3.tsv",
    sep="\t"
)

# ------------------------------------------------------------
# Load ground truth
# ------------------------------------------------------------

ground_truth = pd.read_csv(
    TRAIN_PATH + "train_ground_truth.tsv",
    sep="\t"
)


# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def preprocess_dataframe(df):

    df = df.copy()

    df["name_norm"] = (
        df["business_name"]
        .apply(normalize_text)
    )

    df["address_norm"] = (
        df["business_address"]
        .apply(normalize_text)
    )

    df["country_norm"] = (
        df["country"]
        .apply(normalize_text)
    )

    return df


s1 = preprocess_dataframe(s1)
s2 = preprocess_dataframe(s2)
s3 = preprocess_dataframe(s3)


# ------------------------------------------------------------
# Ground truth
# ------------------------------------------------------------

def parse_matches(x):

    if pd.isna(x) or str(x).strip() == "":
        return []

    return [
        item.strip()
        for item in str(x).split(",")
        if item.strip()
    ]


ground_truth["matched_list"] = (
    ground_truth["matched_entity_ids"]
    .apply(parse_matches)
)

ground_truth_lookup = dict(
    zip(
        ground_truth["source1_entity_id"],
        ground_truth["matched_list"]
    )
)


# ------------------------------------------------------------
# Combined candidate pool
# ------------------------------------------------------------

candidate_pool = pd.concat(
    [
        s2[
            [
                "entity_id",
                "name_norm",
                "address_norm",
                "country_norm"
            ]
        ],
        s3[
            [
                "entity_id",
                "name_norm",
                "address_norm",
                "country_norm"
            ]
        ]
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Candidate lookup
# ------------------------------------------------------------

candidate_lookup = (
    candidate_pool
    .set_index("entity_id")
    [
        [
            "name_norm",
            "address_norm",
            "country_norm"
        ]
    ]
    .to_dict("index")
)


print("===================================")
print("PHASE 7 RESTORE COMPLETE")
print("===================================")

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)
print("Ground truth:", ground_truth.shape)
print("Candidate pool:", candidate_pool.shape)
print("Candidate lookup:", len(candidate_lookup))

PHASE 7 RESTORE COMPLETE
S1: (2206821, 7)
S2: (5034616, 7)
S3: (5285603, 7)
Ground truth: (2206821, 3)
Candidate pool: (10320219, 4)
Candidate lookup: 10320219


In [6]:
# ============================================================
# PHASE 7.1 - BUILD COMPACT ML TRAINING DATASET
# ============================================================

TRAIN_S1_LIMIT = 2000
NEGATIVES_PER_S1 = 5

training_rows = []

print("Building training pairs...")

for i in range(TRAIN_S1_LIMIT):

    row = s1.iloc[i]
    s1_id = row["entity_id"]

    # True matches
    true_matches = set(
        ground_truth_lookup.get(s1_id, [])
    )

    # --------------------------------------------------------
    # Candidate retrieval
    # --------------------------------------------------------
    candidates = set()

    country = row["country_norm"]
    name = row["name_norm"]

    # Exact name
    if name:
        candidates.update(
            exact_name_lookup.get(
                (country, name),
                []
            )
        )

    # Prefix
    if name:
        prefix = (
            name.replace(" ", "")[:3]
        )

        candidates.update(
            prefix_lookup.get(
                (country, prefix),
                []
            )
        )

    # --------------------------------------------------------
    # Positives recovered by blocking
    # --------------------------------------------------------

    recovered_positives = (
        candidates.intersection(true_matches)
    )

    for cid in recovered_positives:

        training_rows.append(
            (
                s1_id,
                cid,
                1
            )
        )

    # --------------------------------------------------------
    # Hard negatives
    # --------------------------------------------------------

    negatives = list(
        candidates - true_matches
    )

    if len(negatives) > NEGATIVES_PER_S1:

        negatives = random.sample(
            negatives,
            NEGATIVES_PER_S1
        )

    for cid in negatives:

        training_rows.append(
            (
                s1_id,
                cid,
                0
            )
        )

    if (i + 1) % 500 == 0:

        print(
            f"Processed "
            f"{i+1:,}/{TRAIN_S1_LIMIT:,}"
        )


training_pairs_df = pd.DataFrame(
    training_rows,
    columns=[
        "source1_entity_id",
        "candidate_entity_id",
        "label"
    ]
)

print("\n===================================")
print("PHASE 7.1 COMPLETE")
print("===================================")

print(
    "Total pairs:",
    len(training_pairs_df)
)

print(
    "Positive:",
    int(
        (training_pairs_df["label"] == 1).sum()
    )
)

print(
    "Negative:",
    int(
        (training_pairs_df["label"] == 0).sum()
    )
)

print(
    "Positive ratio:",
    round(
        training_pairs_df["label"].mean(),
        4
    )
)

Building training pairs...


NameError: name 'exact_name_lookup' is not defined

In [7]:
# ============================================================
# PHASE 7 - BUILD ONLY REQUIRED TRAINING INDEXES
# ============================================================

# Exact name + country
exact_name_lookup = (
    candidate_pool[
        candidate_pool["name_norm"] != ""
    ]
    .groupby(
        ["country_norm", "name_norm"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

# 3-character prefix + country
candidate_pool["name_prefix"] = (
    candidate_pool["name_norm"]
    .str.replace(" ", "", regex=False)
    .str[:3]
)

prefix_lookup = (
    candidate_pool[
        candidate_pool["name_prefix"] != ""
    ]
    .groupby(
        ["country_norm", "name_prefix"],
        sort=False
    )["entity_id"]
    .agg(list)
    .to_dict()
)

print("===================================")
print("PHASE 7 INDEXES READY")
print("===================================")

print(
    "Exact-name groups:",
    len(exact_name_lookup)
)

print(
    "Prefix groups:",
    len(prefix_lookup)
)

PHASE 7 INDEXES READY
Exact-name groups: 7165488
Prefix groups: 48132


In [8]:
# ============================================================
# PHASE 7.1 - BUILD COMPACT ML TRAINING DATASET
# ============================================================

TRAIN_S1_LIMIT = 2000
NEGATIVES_PER_S1 = 5

training_rows = []

print("Building training pairs...")

for i in range(TRAIN_S1_LIMIT):

    row = s1.iloc[i]
    s1_id = row["entity_id"]

    # True matches
    true_matches = set(
        ground_truth_lookup.get(s1_id, [])
    )

    # --------------------------------------------------------
    # Candidate retrieval
    # --------------------------------------------------------
    candidates = set()

    country = row["country_norm"]
    name = row["name_norm"]

    # Exact name
    if name:
        candidates.update(
            exact_name_lookup.get(
                (country, name),
                []
            )
        )

    # Prefix
    if name:
        prefix = (
            name.replace(" ", "")[:3]
        )

        candidates.update(
            prefix_lookup.get(
                (country, prefix),
                []
            )
        )

    # --------------------------------------------------------
    # Positives recovered by blocking
    # --------------------------------------------------------

    recovered_positives = (
        candidates.intersection(true_matches)
    )

    for cid in recovered_positives:

        training_rows.append(
            (
                s1_id,
                cid,
                1
            )
        )

    # --------------------------------------------------------
    # Hard negatives
    # --------------------------------------------------------

    negatives = list(
        candidates - true_matches
    )

    if len(negatives) > NEGATIVES_PER_S1:

        negatives = random.sample(
            negatives,
            NEGATIVES_PER_S1
        )

    for cid in negatives:

        training_rows.append(
            (
                s1_id,
                cid,
                0
            )
        )

    if (i + 1) % 500 == 0:

        print(
            f"Processed "
            f"{i+1:,}/{TRAIN_S1_LIMIT:,}"
        )


training_pairs_df = pd.DataFrame(
    training_rows,
    columns=[
        "source1_entity_id",
        "candidate_entity_id",
        "label"
    ]
)

print("\n===================================")
print("PHASE 7.1 COMPLETE")
print("===================================")

print(
    "Total pairs:",
    len(training_pairs_df)
)

print(
    "Positive:",
    int(
        (training_pairs_df["label"] == 1).sum()
    )
)

print(
    "Negative:",
    int(
        (training_pairs_df["label"] == 0).sum()
    )
)

print(
    "Positive ratio:",
    round(
        training_pairs_df["label"].mean(),
        4
    )
)

Building training pairs...
Processed 500/2,000
Processed 1,000/2,000
Processed 1,500/2,000
Processed 2,000/2,000

PHASE 7.1 COMPLETE
Total pairs: 15398
Positive: 5422
Negative: 9976
Positive ratio: 0.3521


In [10]:
# ============================================================
# PHASE 7.2 - FEATURE ENGINEERING
# ============================================================

from rapidfuzz import fuzz
import re
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Helper: numeric values from address
# ------------------------------------------------------------

def extract_numbers(text):

    if not text:
        return set()

    return set(
        re.findall(
            r"\b\d+\b",
            text
        )
    )


# ------------------------------------------------------------
# Helper: numeric overlap
# ------------------------------------------------------------

def numeric_overlap(a, b):

    nums_a = extract_numbers(a)
    nums_b = extract_numbers(b)

    if not nums_a or not nums_b:
        return 0.0

    return len(
        nums_a.intersection(nums_b)
    ) / len(
        nums_a.union(nums_b)
    )


# ------------------------------------------------------------
# Build feature rows
# ------------------------------------------------------------

feature_rows = []

print("Generating features...")

for idx, pair in enumerate(
    training_pairs_df.itertuples(index=False)
):

    s1_id = pair.source1_entity_id
    candidate_id = pair.candidate_entity_id
    label = pair.label

    # Source 1 record
    s1_row = s1[
        s1["entity_id"] == s1_id
    ].iloc[0]

    # Candidate record
    candidate = candidate_lookup[
        candidate_id
    ]

    name1 = s1_row["name_norm"]
    name2 = candidate["name_norm"]

    address1 = s1_row["address_norm"]
    address2 = candidate["address_norm"]

    country1 = s1_row["country_norm"]
    country2 = candidate["country_norm"]


    # --------------------------------------------------------
    # NAME FEATURES
    # --------------------------------------------------------

    name_ratio = fuzz.ratio(
        name1,
        name2
    )

    name_token_set = fuzz.token_set_ratio(
        name1,
        name2
    )

    name_token_sort = fuzz.token_sort_ratio(
        name1,
        name2
    )

    name_partial = fuzz.partial_ratio(
        name1,
        name2
    )

    exact_name = int(
        name1 == name2 and name1 != ""
    )


    # --------------------------------------------------------
    # ADDRESS FEATURES
    # --------------------------------------------------------

    address_ratio = fuzz.ratio(
        address1,
        address2
    )

    address_token_set = fuzz.token_set_ratio(
        address1,
        address2
    )

    address_partial = fuzz.partial_ratio(
        address1,
        address2
    )

    address_num_overlap = numeric_overlap(
        address1,
        address2
    )


    # --------------------------------------------------------
    # OTHER FEATURES
    # --------------------------------------------------------

    country_match = int(
        country1 == country2
    )

    name_length_diff = abs(
        len(name1) - len(name2)
    )

    address_length_diff = abs(
        len(address1) - len(address2)
    )


    feature_rows.append(
        {
            "name_ratio": name_ratio,
            "name_token_set": name_token_set,
            "name_token_sort": name_token_sort,
            "name_partial": name_partial,
            "exact_name": exact_name,

            "address_ratio": address_ratio,
            "address_token_set": address_token_set,
            "address_partial": address_partial,
            "address_num_overlap": address_num_overlap,

            "country_match": country_match,

            "name_length_diff": name_length_diff,
            "address_length_diff": address_length_diff,

            "label": label
        }
    )


    if (idx + 1) % 2000 == 0:

        print(
            f"Processed "
            f"{idx+1:,}/"
            f"{len(training_pairs_df):,}"
        )


features_df = pd.DataFrame(
    feature_rows
)


print("\n===================================")
print("PHASE 7.2 COMPLETE")
print("===================================")

print(
    "Feature matrix:",
    features_df.shape
)

print("\nColumns:")
print(
    list(features_df.columns)
)

print("\nLabel distribution:")
print(
    features_df["label"]
    .value_counts()
)

Generating features...
Processed 2,000/15,398
Processed 4,000/15,398
Processed 6,000/15,398
Processed 8,000/15,398
Processed 10,000/15,398
Processed 12,000/15,398
Processed 14,000/15,398

PHASE 7.2 COMPLETE
Feature matrix: (15398, 13)

Columns:
['name_ratio', 'name_token_set', 'name_token_sort', 'name_partial', 'exact_name', 'address_ratio', 'address_token_set', 'address_partial', 'address_num_overlap', 'country_match', 'name_length_diff', 'address_length_diff', 'label']

Label distribution:
label
0    9976
1    5422
Name: count, dtype: int64


In [11]:
# ============================================================
# PHASE 8.1 - ML MODEL TRAINING
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# Features and target
# ------------------------------------------------------------

feature_columns = [
    "name_ratio",
    "name_token_set",
    "name_token_sort",
    "name_partial",
    "exact_name",
    "address_ratio",
    "address_token_set",
    "address_partial",
    "address_num_overlap",
    "country_match",
    "name_length_diff",
    "address_length_diff"
]

X = features_df[feature_columns]
y = features_df["label"]


# ------------------------------------------------------------
# Train / validation split
# ------------------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Random Forest...")

model.fit(
    X_train,
    y_train
)

print("Model training complete.")


# ------------------------------------------------------------
# Validation predictions
# ------------------------------------------------------------

y_pred = model.predict(X_val)

print("\n===================================")
print("PHASE 8.1 RESULTS")
print("===================================")

print(
    "Precision:",
    round(
        precision_score(y_val, y_pred),
        4
    )
)

print(
    "Recall:",
    round(
        recall_score(y_val, y_pred),
        4
    )
)

print(
    "F1:",
    round(
        f1_score(y_val, y_pred),
        4
    )
)

print("\nClassification report:")
print(
    classification_report(
        y_val,
        y_pred
    )
)

Training samples: 12318
Validation samples: 3080

Training Random Forest...
Model training complete.

PHASE 8.1 RESULTS
Precision: 0.9881
Recall: 0.9963
F1: 0.9922

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1995
           1       0.99      1.00      0.99      1085

    accuracy                           0.99      3080
   macro avg       0.99      0.99      0.99      3080
weighted avg       0.99      0.99      0.99      3080



In [12]:
# ============================================================
# PHASE 9 - F0.5 SCORE + THRESHOLD OPTIMIZATION
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score
)
import numpy as np


# ------------------------------------------------------------
# Get probability of MATCH
# ------------------------------------------------------------

y_prob = model.predict_proba(
    X_val
)[:, 1]


# ------------------------------------------------------------
# Test multiple thresholds
# ------------------------------------------------------------

thresholds = np.arange(
    0.10,
    0.96,
    0.05
)

results = []

for threshold in thresholds:

    pred = (
        y_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        pred,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val,
        pred,
        beta=0.5,
        zero_division=0
    )

    results.append(
        {
            "threshold": round(
                threshold,
                2
            ),
            "precision": precision,
            "recall": recall,
            "F0.5": f05
        }
    )


threshold_results = pd.DataFrame(
    results
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("===================================")
print("PHASE 9 - F0.5 RESULTS")
print("===================================")

print(
    threshold_results.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Best threshold
# ------------------------------------------------------------

best_row = threshold_results.loc[
    threshold_results["F0.5"].idxmax()
]

BEST_THRESHOLD = float(
    best_row["threshold"]
)

print("\n===================================")
print("BEST THRESHOLD")
print("===================================")

print(
    "Threshold:",
    BEST_THRESHOLD
)

print(
    "Precision:",
    round(
        best_row["precision"],
        4
    )
)

print(
    "Recall:",
    round(
        best_row["recall"],
        4
    )
)

print(
    "F0.5:",
    round(
        best_row["F0.5"],
        4
    )
)

PHASE 9 - F0.5 RESULTS
 threshold  precision   recall     F0.5
      0.10   0.962733 1.000000 0.969962
      0.15   0.971352 1.000000 0.976949
      0.20   0.976598 1.000000 0.981190
      0.25   0.981900 1.000000 0.985468
      0.30   0.982774 0.999078 0.985992
      0.35   0.983651 0.998157 0.986518
      0.40   0.985441 0.998157 0.987958
      0.45   0.986339 0.998157 0.988680
      0.50   0.988117 0.996313 0.989745
      0.55   0.989918 0.995392 0.991008
      0.60   0.989908 0.994470 0.990817
      0.65   0.990817 0.994470 0.991546
      0.70   0.990809 0.993548 0.991356
      0.75   0.991720 0.993548 0.992085
      0.80   0.991713 0.992627 0.991895
      0.85   0.991674 0.988018 0.990941
      0.90   0.993446 0.977880 0.990293
      0.95   0.996154 0.954839 0.987607

BEST THRESHOLD
Threshold: 0.75
Precision: 0.9917
Recall: 0.9935
F0.5: 0.9921


In [13]:
# ==========================================
# PHASE 10 - FINAL MODEL CONFIGURATION
# ==========================================

BEST_THRESHOLD = 0.75

print("===================================")
print("PHASE 10 - MODEL CONFIGURED")
print("===================================")
print("Model: Random Forest")
print("Number of trees:", model.n_estimators)
print("Max depth:", model.max_depth)
print("Minimum leaf size:", model.min_samples_leaf)
print("Class weight:", model.class_weight)
print("Final threshold:", BEST_THRESHOLD)

print("\nValidation performance:")
print("Precision:", round(best_row["precision"], 4))
print("Recall:", round(best_row["recall"], 4))
print("F0.5:", round(best_row["F0.5"], 4))

PHASE 10 - MODEL CONFIGURED
Model: Random Forest
Number of trees: 250
Max depth: 12
Minimum leaf size: 3
Class weight: balanced
Final threshold: 0.75

Validation performance:
Precision: 0.9917
Recall: 0.9935
F0.5: 0.9921


In [14]:
# ==========================================
# PHASE 11 - TEST CANDIDATES + ML PREDICTION
# ==========================================

import os
import numpy as np
import pandas as pd
import re

TEST_S1_PATH = "../dataset/test/test_source1.tsv"
TEST_S2_PATH = "../dataset/test/test_source2.tsv"
TEST_S3_PATH = "../dataset/test/test_source3.tsv"

OUTPUT_DIR = "../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MATCHING_OUT = os.path.join(OUTPUT_DIR, "matching_results.tsv")
CANDIDATE_OUT = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")

BATCH_SIZE = 5000
FINAL_THRESHOLD = BEST_THRESHOLD

print("===================================")
print("PHASE 11 - LOADING TEST DATA")
print("===================================")

test_s1 = preprocess_dataframe(
    pd.read_csv(TEST_S1_PATH, sep="\t", dtype=str)
)

test_s2 = preprocess_dataframe(
    pd.read_csv(TEST_S2_PATH, sep="\t", dtype=str)
)

test_s3 = preprocess_dataframe(
    pd.read_csv(TEST_S3_PATH, sep="\t", dtype=str)
)

print("Test S1:", test_s1.shape)
print("Test S2:", test_s2.shape)
print("Test S3:", test_s3.shape)

# ------------------------------------------------
# Combine S2 + S3
# ------------------------------------------------

test_pool = pd.concat(
    [test_s2, test_s3],
    ignore_index=True
)

print("Combined candidate pool:", test_pool.shape)

# ------------------------------------------------
# Build lightweight lookup dictionaries
# ------------------------------------------------

print("\nBuilding test indexes...")

test_exact_lookup = {}

for idx, name in enumerate(test_pool["name_norm"].values):

    if not name:
        continue

    test_exact_lookup.setdefault(name, []).append(idx)


test_prefix_lookup = {}

for idx, name in enumerate(test_pool["name_norm"].values):

    if not name:
        continue

    prefix = name[:4]

    if prefix:
        test_prefix_lookup.setdefault(prefix, []).append(idx)


print("Exact-name groups:", len(test_exact_lookup))
print("Prefix groups:", len(test_prefix_lookup))

print("\n===================================")
print("TEST INDEXES READY")
print("===================================")

PHASE 11 - LOADING TEST DATA
Test S1: (1732544, 7)
Test S2: (4887273, 7)
Test S3: (4771003, 7)
Combined candidate pool: (9658276, 7)

Building test indexes...
Exact-name groups: 6858570
Prefix groups: 131564

TEST INDEXES READY


In [ ]:
# ==========================================
# PHASE 11C - FAST TEST INFERENCE
# ==========================================

import os
import gc
import time
import re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

# ------------------------------------------
# SETTINGS
# ------------------------------------------

BATCH_SIZE = 5000
FINAL_THRESHOLD = 0.75

# Cheap pre-filter
NAME_PREFILTER = 0.55

MATCHING_OUT = "../output/matching_results.tsv"
CANDIDATE_OUT = "../output/candidate_pairs.tsv"

# ------------------------------------------
# Remove incomplete previous output
# ------------------------------------------

for path in [MATCHING_OUT, CANDIDATE_OUT]:
    if os.path.exists(path):
        os.remove(path)

# Headers
pd.DataFrame(
    columns=["source1_entity_id", "matched_entity_ids"]
).to_csv(
    MATCHING_OUT,
    sep="\t",
    index=False
)

pd.DataFrame(
    columns=["source1_entity_id", "candidate_entity_ids"]
).to_csv(
    CANDIDATE_OUT,
    sep="\t",
    index=False
)

# ------------------------------------------
# Test pool arrays
# ------------------------------------------

pool_ids = test_pool["entity_id"].values
pool_names = test_pool["name_norm"].values
pool_addresses = test_pool["address_norm"].values
pool_countries = test_pool["country_norm"].values

total_s1 = len(test_s1)

start_time = time.time()

print("===================================")
print("PHASE 11C - FAST INFERENCE")
print("===================================")
print("Total S1:", total_s1)
print("Batch size:", BATCH_SIZE)
print("ML threshold:", FINAL_THRESHOLD)
print("Name pre-filter:", NAME_PREFILTER)
print()

# ------------------------------------------
# Process batches
# ------------------------------------------

for batch_start in range(0, total_s1, BATCH_SIZE):

    batch_end = min(
        batch_start + BATCH_SIZE,
        total_s1
    )

    batch = test_s1.iloc[batch_start:batch_end]

    candidate_rows = []
    feature_rows = []
    feature_info = []

    # --------------------------------------
    # Candidate generation
    # --------------------------------------

    for row in batch.itertuples(index=False):

        s1_id = row.entity_id
        name = row.name_norm
        address = row.address_norm
        country = row.country_norm

        indices = set()

        # Exact name
        if name:
            indices.update(
                test_exact_lookup.get(name, [])
            )

            # Prefix
            prefix = name[:4]

            if prefix:
                indices.update(
                    test_prefix_lookup.get(prefix, [])
                )

        # Country filter
        if indices:
            indices = [
                i for i in indices
                if pool_countries[i] == country
            ]

        # ----------------------------------
        # Cheap name pre-filter
        # ----------------------------------

        filtered = []

        if indices:

            for i in indices:

                candidate_name = pool_names[i]

                # Exact name always survives
                if name == candidate_name:
                    filtered.append(i)
                    continue

                # Cheap ratio
                score = fuzz.ratio(
                    name,
                    candidate_name
                ) / 100.0

                if score >= NAME_PREFILTER:
                    filtered.append(i)

        # ----------------------------------
        # Candidate output
        # ----------------------------------

        candidate_ids = [
            pool_ids[i]
            for i in filtered
        ]

        candidate_rows.append({
            "source1_entity_id": s1_id,
            "candidate_entity_ids": ",".join(candidate_ids)
        })

        # ----------------------------------
        # ML features
        # ----------------------------------

        for i in filtered:

            c_name = pool_names[i]
            c_address = pool_addresses[i]

            name_ratio = fuzz.ratio(
                name, c_name
            ) / 100.0

            name_token_set = fuzz.token_set_ratio(
                name, c_name
            ) / 100.0

            name_token_sort = fuzz.token_sort_ratio(
                name, c_name
            ) / 100.0

            name_partial = fuzz.partial_ratio(
                name, c_name
            ) / 100.0

            exact_name = int(
                name == c_name and name != ""
            )

            address_ratio = fuzz.ratio(
                address, c_address
            ) / 100.0

            address_token_set = fuzz.token_set_ratio(
                address, c_address
            ) / 100.0

            address_partial = fuzz.partial_ratio(
                address, c_address
            ) / 100.0

            # Numbers in address
            nums1 = set(
                re.findall(r"\b\d+\b", address)
            )

            nums2 = set(
                re.findall(r"\b\d+\b", c_address)
            )

            if nums1 and nums2:
                address_num_overlap = (
                    len(nums1 & nums2)
                    / len(nums1 | nums2)
                )
            else:
                address_num_overlap = 0.0

            country_match = int(
                country == pool_countries[i]
            )

            name_length_diff = abs(
                len(name) - len(c_name)
            )

            address_length_diff = abs(
                len(address) - len(c_address)
            )

            feature_rows.append([
                name_ratio,
                name_token_set,
                name_token_sort,
                name_partial,
                exact_name,
                address_ratio,
                address_token_set,
                address_partial,
                address_num_overlap,
                country_match,
                name_length_diff,
                address_length_diff
            ])

            feature_info.append(
                (s1_id, i)
            )

    # --------------------------------------
    # ML prediction
    # --------------------------------------

    predictions = {}

    if feature_rows:

        X_batch = pd.DataFrame(
            feature_rows,
            columns=feature_columns
        )

        probabilities = model.predict_proba(
            X_batch
        )[:, 1]

        for info, probability in zip(
            feature_info,
            probabilities
        ):

            if probability >= FINAL_THRESHOLD:

                s1_id, pool_index = info

                predictions.setdefault(
                    s1_id,
                    []
                ).append(
                    pool_ids[pool_index]
                )

        del X_batch
        del probabilities

    # --------------------------------------
    # Matching results
    # --------------------------------------

    matching_rows = []

    for row in candidate_rows:

        s1_id = row["source1_entity_id"]

        matched = predictions.get(
            s1_id,
            []
        )

        matching_rows.append({
            "source1_entity_id": s1_id,
            "matched_entity_ids": ",".join(matched)
        })

    # --------------------------------------
    # Write files
    # --------------------------------------

    pd.DataFrame(
        matching_rows
    ).to_csv(
        MATCHING_OUT,
        sep="\t",
        index=False,
        mode="a",
        header=False
    )

    pd.DataFrame(
        candidate_rows
    ).to_csv(
        CANDIDATE_OUT,
        sep="\t",
        index=False,
        mode="a",
        header=False
    )

    # --------------------------------------
    # Progress
    # --------------------------------------

    processed = batch_end

    elapsed = time.time() - start_time

    rate = processed / max(elapsed, 1)

    remaining = total_s1 - processed

    eta = remaining / max(rate, 0.001)

    print(
        f"Processed {processed:,}/{total_s1:,} "
        f"({processed / total_s1 * 100:.2f}%) | "
        f"Rate: {rate:.1f} S1/s | "
        f"ETA: {eta / 60:.1f} min"
    )

    # Free memory
    del candidate_rows
    del feature_rows
    del feature_info
    del matching_rows
    del predictions

    gc.collect()


print()
print("===================================")
print("PHASE 11C COMPLETE")
print("===================================")
print("Matching file:")
print(MATCHING_OUT)

print("\nCandidate file:")
print(CANDIDATE_OUT)

print(
    "\nTotal time:",
    round(
        (time.time() - start_time) / 60,
        2
    ),
    "minutes"
)

PHASE 11C - FAST INFERENCE
Total S1: 1732544
Batch size: 5000
ML threshold: 0.75
Name pre-filter: 0.55



In [3]:
# ==========================================
# PHASE 11C - FAST TEST INFERENCE
# ==========================================

import os
import gc
import time
import re
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

# ------------------------------------------
# SETTINGS
# ------------------------------------------

BATCH_SIZE = 5000
FINAL_THRESHOLD = 0.75

# Cheap pre-filter
NAME_PREFILTER = 0.55

MATCHING_OUT = "../output/matching_results.tsv"
CANDIDATE_OUT = "../output/candidate_pairs.tsv"

# ------------------------------------------
# Remove incomplete previous output
# ------------------------------------------

for path in [MATCHING_OUT, CANDIDATE_OUT]:
    if os.path.exists(path):
        os.remove(path)

# Headers
pd.DataFrame(
    columns=["source1_entity_id", "matched_entity_ids"]
).to_csv(
    MATCHING_OUT,
    sep="\t",
    index=False
)

pd.DataFrame(
    columns=["source1_entity_id", "candidate_entity_ids"]
).to_csv(
    CANDIDATE_OUT,
    sep="\t",
    index=False
)

# ------------------------------------------
# Test pool arrays
# ------------------------------------------

pool_ids = test_pool["entity_id"].values
pool_names = test_pool["name_norm"].values
pool_addresses = test_pool["address_norm"].values
pool_countries = test_pool["country_norm"].values

total_s1 = len(test_s1)

start_time = time.time()

print("===================================")
print("PHASE 11C - FAST INFERENCE")
print("===================================")
print("Total S1:", total_s1)
print("Batch size:", BATCH_SIZE)
print("ML threshold:", FINAL_THRESHOLD)
print("Name pre-filter:", NAME_PREFILTER)
print()

# ------------------------------------------
# Process batches
# ------------------------------------------

for batch_start in range(0, total_s1, BATCH_SIZE):

    batch_end = min(
        batch_start + BATCH_SIZE,
        total_s1
    )

    batch = test_s1.iloc[batch_start:batch_end]

    candidate_rows = []
    feature_rows = []
    feature_info = []

    # --------------------------------------
    # Candidate generation
    # --------------------------------------

    for row in batch.itertuples(index=False):

        s1_id = row.entity_id
        name = row.name_norm
        address = row.address_norm
        country = row.country_norm

        indices = set()

        # Exact name
        if name:
            indices.update(
                test_exact_lookup.get(name, [])
            )

            # Prefix
            prefix = name[:4]

            if prefix:
                indices.update(
                    test_prefix_lookup.get(prefix, [])
                )

        # Country filter
        if indices:
            indices = [
                i for i in indices
                if pool_countries[i] == country
            ]

        # ----------------------------------
        # Cheap name pre-filter
        # ----------------------------------

        filtered = []

        if indices:

            for i in indices:

                candidate_name = pool_names[i]

                # Exact name always survives
                if name == candidate_name:
                    filtered.append(i)
                    continue

                # Cheap ratio
                score = fuzz.ratio(
                    name,
                    candidate_name
                ) / 100.0

                if score >= NAME_PREFILTER:
                    filtered.append(i)

        # ----------------------------------
        # Candidate output
        # ----------------------------------

        candidate_ids = [
            pool_ids[i]
            for i in filtered
        ]

        candidate_rows.append({
            "source1_entity_id": s1_id,
            "candidate_entity_ids": ",".join(candidate_ids)
        })

        # ----------------------------------
        # ML features
        # ----------------------------------

        for i in filtered:

            c_name = pool_names[i]
            c_address = pool_addresses[i]

            name_ratio = fuzz.ratio(
                name, c_name
            ) / 100.0

            name_token_set = fuzz.token_set_ratio(
                name, c_name
            ) / 100.0

            name_token_sort = fuzz.token_sort_ratio(
                name, c_name
            ) / 100.0

            name_partial = fuzz.partial_ratio(
                name, c_name
            ) / 100.0

            exact_name = int(
                name == c_name and name != ""
            )

            address_ratio = fuzz.ratio(
                address, c_address
            ) / 100.0

            address_token_set = fuzz.token_set_ratio(
                address, c_address
            ) / 100.0

            address_partial = fuzz.partial_ratio(
                address, c_address
            ) / 100.0

            # Numbers in address
            nums1 = set(
                re.findall(r"\b\d+\b", address)
            )

            nums2 = set(
                re.findall(r"\b\d+\b", c_address)
            )

            if nums1 and nums2:
                address_num_overlap = (
                    len(nums1 & nums2)
                    / len(nums1 | nums2)
                )
            else:
                address_num_overlap = 0.0

            country_match = int(
                country == pool_countries[i]
            )

            name_length_diff = abs(
                len(name) - len(c_name)
            )

            address_length_diff = abs(
                len(address) - len(c_address)
            )

            feature_rows.append([
                name_ratio,
                name_token_set,
                name_token_sort,
                name_partial,
                exact_name,
                address_ratio,
                address_token_set,
                address_partial,
                address_num_overlap,
                country_match,
                name_length_diff,
                address_length_diff
            ])

            feature_info.append(
                (s1_id, i)
            )

    # --------------------------------------
    # ML prediction
    # --------------------------------------

    predictions = {}

    if feature_rows:

        X_batch = pd.DataFrame(
            feature_rows,
            columns=feature_columns
        )

        probabilities = model.predict_proba(
            X_batch
        )[:, 1]

        for info, probability in zip(
            feature_info,
            probabilities
        ):

            if probability >= FINAL_THRESHOLD:

                s1_id, pool_index = info

                predictions.setdefault(
                    s1_id,
                    []
                ).append(
                    pool_ids[pool_index]
                )

        del X_batch
        del probabilities

    # --------------------------------------
    # Matching results
    # --------------------------------------

    matching_rows = []

    for row in candidate_rows:

        s1_id = row["source1_entity_id"]

        matched = predictions.get(
            s1_id,
            []
        )

        matching_rows.append({
            "source1_entity_id": s1_id,
            "matched_entity_ids": ",".join(matched)
        })

    # --------------------------------------
    # Write files
    # --------------------------------------

    pd.DataFrame(
        matching_rows
    ).to_csv(
        MATCHING_OUT,
        sep="\t",
        index=False,
        mode="a",
        header=False
    )

    pd.DataFrame(
        candidate_rows
    ).to_csv(
        CANDIDATE_OUT,
        sep="\t",
        index=False,
        mode="a",
        header=False
    )

    # --------------------------------------
    # Progress
    # --------------------------------------

    processed = batch_end

    elapsed = time.time() - start_time

    rate = processed / max(elapsed, 1)

    remaining = total_s1 - processed

    eta = remaining / max(rate, 0.001)

    print(
        f"Processed {processed:,}/{total_s1:,} "
        f"({processed / total_s1 * 100:.2f}%) | "
        f"Rate: {rate:.1f} S1/s | "
        f"ETA: {eta / 60:.1f} min"
    )

    # Free memory
    del candidate_rows
    del feature_rows
    del feature_info
    del matching_rows
    del predictions

    gc.collect()


print()
print("===================================")
print("PHASE 11C COMPLETE")
print("===================================")
print("Matching file:")
print(MATCHING_OUT)

print("\nCandidate file:")
print(CANDIDATE_OUT)

print(
    "\nTotal time:",
    round(
        (time.time() - start_time) / 60,
        2
    ),
    "minutes"
)

NameError: name 'test_pool' is not defined

In [4]:
# ==========================================
# RESTORE TEST DATA + INDEXES
# ==========================================

import pandas as pd
import numpy as np
import re
import unicodedata
import os
import gc
from rapidfuzz import fuzz

TEST_PATH = "../dataset/test/"

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_dataframe(df):
    df = df.copy()

    df["name_norm"] = df["business_name"].apply(
        normalize_text
    )

    df["address_norm"] = df["business_address"].apply(
        normalize_text
    )

    df["country_norm"] = df["country"].apply(
        normalize_text
    )

    return df


print("===================================")
print("RESTORING TEST DATA")
print("===================================")

test_s1 = preprocess_dataframe(
    pd.read_csv(
        TEST_PATH + "test_source1.tsv",
        sep="\t",
        dtype=str
    )
)

test_s2 = preprocess_dataframe(
    pd.read_csv(
        TEST_PATH + "test_source2.tsv",
        sep="\t",
        dtype=str
    )
)

test_s3 = preprocess_dataframe(
    pd.read_csv(
        TEST_PATH + "test_source3.tsv",
        sep="\t",
        dtype=str
    )
)

print("S1:", test_s1.shape)
print("S2:", test_s2.shape)
print("S3:", test_s3.shape)


# ------------------------------------------
# Combine S2 + S3
# ------------------------------------------

test_pool = pd.concat(
    [test_s2, test_s3],
    ignore_index=True
)

print("Combined pool:", test_pool.shape)


# ------------------------------------------
# Exact name index
# ------------------------------------------

print("\nBuilding exact-name index...")

test_exact_lookup = {}

for idx, name in enumerate(
    test_pool["name_norm"].values
):

    if name:
        test_exact_lookup.setdefault(
            name, []
        ).append(idx)

print(
    "Exact-name groups:",
    len(test_exact_lookup)
)


# ------------------------------------------
# Prefix index
# ------------------------------------------

print("Building prefix index...")

test_prefix_lookup = {}

for idx, name in enumerate(
    test_pool["name_norm"].values
):

    if name:
        prefix = name[:4]

        if prefix:
            test_prefix_lookup.setdefault(
                prefix, []
            ).append(idx)

print(
    "Prefix groups:",
    len(test_prefix_lookup)
)


print()
print("===================================")
print("TEST RESTORE COMPLETE")
print("===================================")
print("S1:", test_s1.shape)
print("S2:", test_s2.shape)
print("S3:", test_s3.shape)
print("Pool:", test_pool.shape)
print("Exact groups:", len(test_exact_lookup))
print("Prefix groups:", len(test_prefix_lookup))

RESTORING TEST DATA
S1: (1732544, 7)
S2: (4887273, 7)
S3: (4771003, 7)
Combined pool: (9658276, 7)

Building exact-name index...
Exact-name groups: 6858570
Building prefix index...
Prefix groups: 131564

TEST RESTORE COMPLETE
S1: (1732544, 7)
S2: (4887273, 7)
S3: (4771003, 7)
Pool: (9658276, 7)
Exact groups: 6858570
Prefix groups: 131564


In [ ]:
# ==========================================
# STEP 2 - RESTORE RANDOM FOREST MODEL
# ==========================================

import pandas as pd
import numpy as np
import re
import gc

from rapidfuzz import fuzz
from sklearn.ensemble import RandomForestClassifier

TRAIN_PATH = "../dataset/train/"

TRAIN_S1_LIMIT = 2000
NEGATIVES_PER_S1 = 5

print("===================================")
print("STEP 2 - LOADING TRAIN DATA")
print("===================================")

# ------------------------------------------
# Load training data
# ------------------------------------------

train_s1 = preprocess_dataframe(
    pd.read_csv(
        TRAIN_PATH + "train_source1.tsv",
        sep="\t",
        dtype=str
    )
)

train_s2 = preprocess_dataframe(
    pd.read_csv(
        TRAIN_PATH + "train_source2.tsv",
        sep="\t",
        dtype=str
    )
)

train_s3 = preprocess_dataframe(
    pd.read_csv(
        TRAIN_PATH + "train_source3.tsv",
        sep="\t",
        dtype=str
    )
)

ground_truth = pd.read_csv(
    TRAIN_PATH + "train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

print("S1:", train_s1.shape)
print("S2:", train_s2.shape)
print("S3:", train_s3.shape)


# ------------------------------------------
# Combine S2 + S3
# ------------------------------------------

train_pool = pd.concat(
    [train_s2, train_s3],
    ignore_index=True
)

print("Training pool:", train_pool.shape)


# ------------------------------------------
# Ground truth
# ------------------------------------------

def parse_matches(x):
    if pd.isna(x) or str(x).strip() == "":
        return []

    return [
        item.strip()
        for item in str(x).split(",")
        if item.strip()
    ]

ground_truth["matched_list"] = (
    ground_truth["matched_entity_ids"]
    .apply(parse_matches)
)

gt_lookup = dict(
    zip(
        ground_truth["source1_entity_id"],
        ground_truth["matched_list"]
    )
)


# ------------------------------------------
# Training indexes
# ------------------------------------------

print("\nBuilding training indexes...")

train_exact_lookup = {}

for idx, name in enumerate(
    train_pool["name_norm"].values
):
    if name:
        train_exact_lookup.setdefault(
            name, []
        ).append(idx)


train_prefix_lookup = {}

for idx, name in enumerate(
    train_pool["name_norm"].values
):
    if name:
        prefix = name[:4]

        if prefix:
            train_prefix_lookup.setdefault(
                prefix, []
            ).append(idx)


print(
    "Exact groups:",
    len(train_exact_lookup)
)

print(
    "Prefix groups:",
    len(train_prefix_lookup)
)


# ------------------------------------------
# Build training pairs
# ------------------------------------------

print("\nBuilding training pairs...")

training_rows = []

sample_s1 = train_s1.iloc[:TRAIN_S1_LIMIT]

for count, (_, row) in enumerate(
    sample_s1.iterrows(),
    start=1
):

    s1_id = row["entity_id"]
    name = row["name_norm"]
    country = row["country_norm"]

    true_ids = set(
        gt_lookup.get(s1_id, [])
    )

    candidate_indices = set()

    if name:
        candidate_indices.update(
            train_exact_lookup.get(
                name, []
            )
        )

        prefix = name[:4]

        if prefix:
            candidate_indices.update(
                train_prefix_lookup.get(
                    prefix, []
                )
            )

    candidate_indices = [
        idx for idx in candidate_indices
        if train_pool.iloc[idx]["country_norm"]
        == country
    ]

    candidate_ids = [
        train_pool.iloc[idx]["entity_id"]
        for idx in candidate_indices
    ]

    # Keep all retrieved candidates
    for cid in candidate_ids:

        training_rows.append({
            "source1_entity_id": s1_id,
            "candidate_entity_id": cid,
            "label": int(cid in true_ids)
        })

    # --------------------------------------
    # Controlled negatives
    # --------------------------------------

    negative_ids = [
        cid for cid in candidate_ids
        if cid not in true_ids
    ]

    negative_ids = negative_ids[
        :NEGATIVES_PER_S1
    ]

    existing_ids = set(candidate_ids)

    for cid in negative_ids:

        if cid not in existing_ids:

            training_rows.append({
                "source1_entity_id": s1_id,
                "candidate_entity_id": cid,
                "label": 0
            })

    if count % 500 == 0:
        print(
            f"Processed {count:,}/{TRAIN_S1_LIMIT:,}"
        )


training_pairs_df = pd.DataFrame(
    training_rows
)

print("\nTraining pairs:", len(training_pairs_df))
print(
    "Positive:",
    int(training_pairs_df["label"].sum())
)
print(
    "Negative:",
    int(
        (training_pairs_df["label"] == 0).sum()
    )
)


# ------------------------------------------
# Fast lookup for S1
# ------------------------------------------

s1_lookup = train_s1.set_index(
    "entity_id"
)

pool_lookup = train_pool.set_index(
    "entity_id"
)


# ------------------------------------------
# Feature engineering
# ------------------------------------------

print("\nBuilding features...")

feature_rows = []

for count, row in enumerate(
    training_pairs_df.itertuples(index=False),
    start=1
):

    s1_row = s1_lookup.loc[
        row.source1_entity_id
    ]

    c_row = pool_lookup.loc[
        row.candidate_entity_id
    ]

    name1 = s1_row["name_norm"]
    name2 = c_row["name_norm"]

    address1 = s1_row["address_norm"]
    address2 = c_row["address_norm"]

    country1 = s1_row["country_norm"]
    country2 = c_row["country_norm"]

    nums1 = set(
        re.findall(r"\b\d+\b", address1)
    )

    nums2 = set(
        re.findall(r"\b\d+\b", address2)
    )

    if nums1 and nums2:
        num_overlap = (
            len(nums1 & nums2)
            / len(nums1 | nums2)
        )
    else:
        num_overlap = 0.0

    feature_rows.append([
        fuzz.ratio(name1, name2) / 100,
        fuzz.token_set_ratio(name1, name2) / 100,
        fuzz.token_sort_ratio(name1, name2) / 100,
        fuzz.partial_ratio(name1, name2) / 100,
        int(
            name1 == name2 and name1 != ""
        ),
        fuzz.ratio(address1, address2) / 100,
        fuzz.token_set_ratio(
            address1,
            address2
        ) / 100,
        fuzz.partial_ratio(
            address1,
            address2
        ) / 100,
        num_overlap,
        int(country1 == country2),
        abs(len(name1) - len(name2)),
        abs(len(address1) - len(address2)),
        row.label
    ])

    if count % 5000 == 0:
        print(
            f"Features: {count:,}/"
            f"{len(training_pairs_df):,}"
        )


feature_columns = [
    "name_ratio",
    "name_token_set",
    "name_token_sort",
    "name_partial",
    "exact_name",
    "address_ratio",
    "address_token_set",
    "address_partial",
    "address_num_overlap",
    "country_match",
    "name_length_diff",
    "address_length_diff"
]

features_df = pd.DataFrame(
    feature_rows,
    columns=feature_columns + ["label"]
)


# ------------------------------------------
# Train model
# ------------------------------------------

print("\nTraining Random Forest...")

X = features_df[feature_columns]
y = features_df["label"]

model = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)

BEST_THRESHOLD = 0.75

print()
print("===================================")
print("STEP 2 COMPLETE")
print("===================================")
print("Feature matrix:", features_df.shape)
print("Training samples:", len(X))
print("Positive:", int(y.sum()))
print("Negative:", int((y == 0).sum()))
print("Trees:", model.n_estimators)
print("Max depth:", model.max_depth)
print("Final threshold:", BEST_THRESHOLD)

STEP 2 - LOADING TRAIN DATA
S1: (2206821, 7)
S2: (5034616, 7)
S3: (5285603, 7)
Training pool: (10320219, 7)

Building training indexes...
Exact groups: 7153276
Prefix groups: 140358

Building training pairs...
Processed 500/2,000
Processed 1,000/2,000


In [1]:
print("Kernel is working")

Kernel is working


In [4]:
import os

print("Checking for saved model files...\n")

for root, dirs, files in os.walk(".."):
    for file in files:
        if file.endswith((".pkl", ".joblib", ".pickle")):
            print(os.path.join(root, file))

Checking for saved model files...



In [5]:
# ==========================================
# STEP 2 - LIGHTWEIGHT MODEL RESTORATION
# ==========================================

import pandas as pd
import numpy as np
import re
from rapidfuzz import fuzz
from sklearn.ensemble import RandomForestClassifier

TRAIN_PATH = "../dataset/train/"

TRAIN_S1_LIMIT = 1000
NEGATIVES_PER_S1 = 5

print("===================================")
print("STEP 2 - LIGHTWEIGHT MODEL")
print("===================================")

# ------------------------------------------
# Load only the required training files
# ------------------------------------------

s1_train = pd.read_csv(
    TRAIN_PATH + "train_source1.tsv",
    sep="\t",
    dtype=str
)

s2_train = pd.read_csv(
    TRAIN_PATH + "train_source2.tsv",
    sep="\t",
    dtype=str
)

s3_train = pd.read_csv(
    TRAIN_PATH + "train_source3.tsv",
    sep="\t",
    dtype=str
)

gt_train = pd.read_csv(
    TRAIN_PATH + "train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

# Normalize only the sampled S1 and all S2/S3
s1_train = preprocess_dataframe(s1_train.iloc[:TRAIN_S1_LIMIT])
s2_train = preprocess_dataframe(s2_train)
s3_train = preprocess_dataframe(s3_train)

print("Sample S1:", s1_train.shape)
print("S2:", s2_train.shape)
print("S3:", s3_train.shape)


# ------------------------------------------
# Ground truth
# ------------------------------------------

def parse_matches(x):
    if pd.isna(x) or str(x).strip() == "":
        return []

    return [
        x.strip()
        for x in str(x).split(",")
        if x.strip()
    ]

gt_train["matched_list"] = (
    gt_train["matched_entity_ids"]
    .apply(parse_matches)
)

gt_lookup = dict(
    zip(
        gt_train["source1_entity_id"],
        gt_train["matched_list"]
    )
)


# ------------------------------------------
# Create SMALL candidate lookup
# ------------------------------------------

print("\nBuilding lightweight indexes...")

train_pool = pd.concat(
    [s2_train, s3_train],
    ignore_index=True
)

exact_lookup = {}

for idx, name in enumerate(
    train_pool["name_norm"].values
):

    if name:
        exact_lookup.setdefault(
            name, []
        ).append(idx)


prefix_lookup = {}

for idx, name in enumerate(
    train_pool["name_norm"].values
):

    if name:
        prefix = name[:4]

        if prefix:
            prefix_lookup.setdefault(
                prefix, []
            ).append(idx)


print("Exact groups:", len(exact_lookup))
print("Prefix groups:", len(prefix_lookup))


# ------------------------------------------
# Build training pairs
# ------------------------------------------

print("\nBuilding training pairs...")

pool_lookup = train_pool.set_index(
    "entity_id"
)

training_rows = []

for counter, row in enumerate(
    s1_train.itertuples(index=False),
    start=1
):

    s1_id = row.entity_id
    name = row.name_norm
    country = row.country_norm

    true_ids = set(
        gt_lookup.get(
            s1_id,
            []
        )
    )

    candidates = set()

    if name:

        candidates.update(
            exact_lookup.get(
                name,
                []
            )
        )

        prefix = name[:4]

        if prefix:
            candidates.update(
                prefix_lookup.get(
                    prefix,
                    []
                )
            )

    # Country filter
    candidates = [
        idx for idx in candidates
        if train_pool.iloc[idx]["country_norm"]
        == country
    ]

    candidate_ids = [
        train_pool.iloc[idx]["entity_id"]
        for idx in candidates
    ]

    # Positives
    for cid in candidate_ids:

        training_rows.append(
            (
                s1_id,
                cid,
                int(cid in true_ids)
            )
        )

    # --------------------------------------
    # Extra negatives
    # --------------------------------------

    negatives = [
        cid for cid in candidate_ids
        if cid not in true_ids
    ][:NEGATIVES_PER_S1]

    for cid in negatives:

        training_rows.append(
            (
                s1_id,
                cid,
                0
            )
        )

    if counter % 200 == 0:
        print(
            f"Processed {counter:,}/{TRAIN_S1_LIMIT:,}"
        )


training_pairs = pd.DataFrame(
    training_rows,
    columns=[
        "source1_entity_id",
        "candidate_entity_id",
        "label"
    ]
)

print(
    "\nTraining pairs:",
    len(training_pairs)
)

print(
    "Positive:",
    int(training_pairs["label"].sum())
)

print(
    "Negative:",
    int(
        (training_pairs["label"] == 0).sum()
    )
)


# ------------------------------------------
# Feature generation
# ------------------------------------------

print("\nBuilding features...")

s1_lookup = s1_train.set_index(
    "entity_id"
)

features = []

for row in training_pairs.itertuples(
    index=False
):

    a = s1_lookup.loc[
        row.source1_entity_id
    ]

    b = pool_lookup.loc[
        row.candidate_entity_id
    ]

    n1 = a["name_norm"]
    n2 = b["name_norm"]

    ad1 = a["address_norm"]
    ad2 = b["address_norm"]

    c1 = a["country_norm"]
    c2 = b["country_norm"]

    nums1 = set(
        re.findall(r"\b\d+\b", ad1)
    )

    nums2 = set(
        re.findall(r"\b\d+\b", ad2)
    )

    if nums1 and nums2:
        num_overlap = (
            len(nums1 & nums2)
            / len(nums1 | nums2)
        )
    else:
        num_overlap = 0.0

    features.append([
        fuzz.ratio(n1, n2) / 100,
        fuzz.token_set_ratio(n1, n2) / 100,
        fuzz.token_sort_ratio(n1, n2) / 100,
        fuzz.partial_ratio(n1, n2) / 100,
        int(n1 == n2 and n1 != ""),
        fuzz.ratio(ad1, ad2) / 100,
        fuzz.token_set_ratio(ad1, ad2) / 100,
        fuzz.partial_ratio(ad1, ad2) / 100,
        num_overlap,
        int(c1 == c2),
        abs(len(n1) - len(n2)),
        abs(len(ad1) - len(ad2)),
        row.label
    ])


feature_columns = [
    "name_ratio",
    "name_token_set",
    "name_token_sort",
    "name_partial",
    "exact_name",
    "address_ratio",
    "address_token_set",
    "address_partial",
    "address_num_overlap",
    "country_match",
    "name_length_diff",
    "address_length_diff"
]

features_df = pd.DataFrame(
    features,
    columns=feature_columns + ["label"]
)


# ------------------------------------------
# Train Random Forest
# ------------------------------------------

print("\nTraining Random Forest...")

X = features_df[feature_columns]
y = features_df["label"]

model = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)

BEST_THRESHOLD = 0.75

print()
print("===================================")
print("STEP 2 COMPLETE")
print("===================================")
print("Feature matrix:", features_df.shape)
print("Training samples:", len(X))
print("Positive:", int(y.sum()))
print("Negative:", int((y == 0).sum()))
print("Threshold:", BEST_THRESHOLD)

STEP 2 - LIGHTWEIGHT MODEL


NameError: name 'preprocess_dataframe' is not defined

In [7]:
# ==========================================
# STEP 2 - LIGHTWEIGHT MODEL RESTORATION
# SELF-CONTAINED VERSION
# ==========================================

import pandas as pd
import numpy as np
import re
import unicodedata
from rapidfuzz import fuzz
from sklearn.ensemble import RandomForestClassifier

TRAIN_PATH = "../dataset/train/"

TRAIN_S1_LIMIT = 1000
NEGATIVES_PER_S1 = 5


# ==========================================
# NORMALIZATION
# ==========================================

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def preprocess_dataframe(df):

    df = df.copy()

    df["name_norm"] = (
        df["business_name"]
        .apply(normalize_text)
    )

    df["address_norm"] = (
        df["business_address"]
        .apply(normalize_text)
    )

    df["country_norm"] = (
        df["country"]
        .apply(normalize_text)
    )

    return df


print("===================================")
print("STEP 2 - LIGHTWEIGHT MODEL")
print("===================================")


# ==========================================
# LOAD TRAINING DATA
# ==========================================

s1_train = pd.read_csv(
    TRAIN_PATH + "train_source1.tsv",
    sep="\t",
    dtype=str
)

s2_train = pd.read_csv(
    TRAIN_PATH + "train_source2.tsv",
    sep="\t",
    dtype=str
)

s3_train = pd.read_csv(
    TRAIN_PATH + "train_source3.tsv",
    sep="\t",
    dtype=str
)

gt_train = pd.read_csv(
    TRAIN_PATH + "train_ground_truth.tsv",
    sep="\t",
    dtype=str
)


# Only 1000 S1
s1_train = preprocess_dataframe(
    s1_train.iloc[:TRAIN_S1_LIMIT]
)

s2_train = preprocess_dataframe(
    s2_train
)

s3_train = preprocess_dataframe(
    s3_train
)


print("Sample S1:", s1_train.shape)
print("S2:", s2_train.shape)
print("S3:", s3_train.shape)


# ==========================================
# GROUND TRUTH
# ==========================================

def parse_matches(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        item.strip()
        for item in x.split(",")
        if item.strip()
    ]


gt_train["matched_list"] = (
    gt_train["matched_entity_ids"]
    .apply(parse_matches)
)

gt_lookup = dict(
    zip(
        gt_train["source1_entity_id"],
        gt_train["matched_list"]
    )
)


# ==========================================
# COMBINE S2 + S3
# ==========================================

train_pool = pd.concat(
    [s2_train, s3_train],
    ignore_index=True
)

print(
    "Training pool:",
    train_pool.shape
)


# ==========================================
# INDEXES
# ==========================================

print("\nBuilding lightweight indexes...")


exact_lookup = {}

for idx, name in enumerate(
    train_pool["name_norm"].values
):

    if name:

        exact_lookup.setdefault(
            name,
            []
        ).append(idx)


prefix_lookup = {}

for idx, name in enumerate(
    train_pool["name_norm"].values
):

    if name:

        prefix = name[:4]

        if prefix:

            prefix_lookup.setdefault(
                prefix,
                []
            ).append(idx)


print(
    "Exact groups:",
    len(exact_lookup)
)

print(
    "Prefix groups:",
    len(prefix_lookup)
)


# ==========================================
# BUILD TRAINING PAIRS
# ==========================================

print("\nBuilding training pairs...")

training_rows = []


for counter, row in enumerate(
    s1_train.itertuples(index=False),
    start=1
):

    s1_id = row.entity_id

    name = row.name_norm

    country = row.country_norm

    true_ids = set(
        gt_lookup.get(
            s1_id,
            []
        )
    )

    candidates = set()


    # Exact name
    if name:

        candidates.update(
            exact_lookup.get(
                name,
                []
            )
        )


        # Prefix
        prefix = name[:4]

        if prefix:

            candidates.update(
                prefix_lookup.get(
                    prefix,
                    []
                )
            )


    # Country filter
    candidates = [
        idx
        for idx in candidates
        if train_pool.iloc[idx]["country_norm"]
        == country
    ]


    candidate_ids = [
        train_pool.iloc[idx]["entity_id"]
        for idx in candidates
    ]


    # --------------------------------------
    # Add candidates
    # --------------------------------------

    for cid in candidate_ids:

        training_rows.append({

            "source1_entity_id": s1_id,

            "candidate_entity_id": cid,

            "label": int(
                cid in true_ids
            )

        })


    # --------------------------------------
    # Controlled negatives
    # --------------------------------------

    negatives = [
        cid
        for cid in candidate_ids
        if cid not in true_ids
    ][:NEGATIVES_PER_S1]


    for cid in negatives:

        training_rows.append({

            "source1_entity_id": s1_id,

            "candidate_entity_id": cid,

            "label": 0

        })


    if counter % 200 == 0:

        print(
            f"Processed "
            f"{counter:,}/"
            f"{TRAIN_S1_LIMIT:,}"
        )


training_pairs = pd.DataFrame(
    training_rows
)

print(
    "\nTraining pairs:",
    len(training_pairs)
)

print(
    "Positive:",
    int(
        training_pairs["label"].sum()
    )
)

print(
    "Negative:",
    int(
        (training_pairs["label"] == 0).sum()
    )
)


# ==========================================
# LOOKUPS FOR FEATURES
# ==========================================

s1_lookup = s1_train.set_index(
    "entity_id"
)

pool_lookup = train_pool.set_index(
    "entity_id"
)


# ==========================================
# FEATURE ENGINEERING
# ==========================================

print("\nBuilding features...")

feature_rows = []


for row in training_pairs.itertuples(
    index=False
):

    a = s1_lookup.loc[
        row.source1_entity_id
    ]

    b = pool_lookup.loc[
        row.candidate_entity_id
    ]


    name1 = a["name_norm"]
    name2 = b["name_norm"]

    address1 = a["address_norm"]
    address2 = b["address_norm"]

    country1 = a["country_norm"]
    country2 = b["country_norm"]


    nums1 = set(
        re.findall(
            r"\b\d+\b",
            address1
        )
    )

    nums2 = set(
        re.findall(
            r"\b\d+\b",
            address2
        )
    )


    if nums1 and nums2:

        number_overlap = (
            len(nums1 & nums2)
            /
            len(nums1 | nums2)
        )

    else:

        number_overlap = 0.0


    feature_rows.append([

        fuzz.ratio(
            name1,
            name2
        ) / 100,

        fuzz.token_set_ratio(
            name1,
            name2
        ) / 100,

        fuzz.token_sort_ratio(
            name1,
            name2
        ) / 100,

        fuzz.partial_ratio(
            name1,
            name2
        ) / 100,

        int(
            name1 == name2
            and name1 != ""
        ),

        fuzz.ratio(
            address1,
            address2
        ) / 100,

        fuzz.token_set_ratio(
            address1,
            address2
        ) / 100,

        fuzz.partial_ratio(
            address1,
            address2
        ) / 100,

        number_overlap,

        int(
            country1 == country2
        ),

        abs(
            len(name1)
            -
            len(name2)
        ),

        abs(
            len(address1)
            -
            len(address2)
        ),

        row.label
    ])


# ==========================================
# FEATURE DATAFRAME
# ==========================================

feature_columns = [

    "name_ratio",

    "name_token_set",

    "name_token_sort",

    "name_partial",

    "exact_name",

    "address_ratio",

    "address_token_set",

    "address_partial",

    "address_num_overlap",

    "country_match",

    "name_length_diff",

    "address_length_diff"

]


features_df = pd.DataFrame(

    feature_rows,

    columns=feature_columns + [
        "label"
    ]

)


print(
    "Feature matrix:",
    features_df.shape
)


# ==========================================
# TRAIN RANDOM FOREST
# ==========================================

print("\nTraining Random Forest...")


X = features_df[
    feature_columns
]

y = features_df[
    "label"
]


model = RandomForestClassifier(

    n_estimators=250,

    max_depth=12,

    min_samples_leaf=3,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1
)


model.fit(
    X,
    y
)


BEST_THRESHOLD = 0.75


# ==========================================
# COMPLETE
# ==========================================

print()

print(
    "==================================="
)

print(
    "STEP 2 COMPLETE"
)

print(
    "==================================="
)

print(
    "Feature matrix:",
    features_df.shape
)

print(
    "Training samples:",
    len(X)
)

print(
    "Positive:",
    int(y.sum())
)

print(
    "Negative:",
    int(
        (y == 0).sum()
    )
)

print(
    "Trees:",
    model.n_estimators
)

print(
    "Max depth:",
    model.max_depth
)

print(
    "Final threshold:",
    BEST_THRESHOLD
)

STEP 2 - LIGHTWEIGHT MODEL
Sample S1: (1000, 7)
S2: (5034616, 7)
S3: (5285603, 7)
Training pool: (10320219, 7)

Building lightweight indexes...
Exact groups: 7153276
Prefix groups: 140358

Building training pairs...
Processed 200/1,000
Processed 400/1,000
Processed 600/1,000
Processed 800/1,000
Processed 1,000/1,000

Training pairs: 6824403
Positive: 2696
Negative: 6821707

Building features...
Feature matrix: (6824403, 13)

Training Random Forest...

STEP 2 COMPLETE
Feature matrix: (6824403, 13)
Training samples: 6824403
Positive: 2696
Negative: 6821707
Trees: 250
Max depth: 12
Final threshold: 0.75


In [8]:
print("Model:", type(model).__name__)
print("Test S1:", len(test_s1))
print("Test pool:", len(test_pool))
print("Threshold:", BEST_THRESHOLD)

Model: RandomForestClassifier


NameError: name 'test_s1' is not defined

In [2]:
# ==========================================
# RESTORE TEST DATA FOR FINAL INFERENCE
# ==========================================

import pandas as pd
import numpy as np
import re
import unicodedata

TEST_PATH = "../dataset/test/"

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def preprocess_dataframe(df):
    df = df.copy()

    df["name_norm"] = (
        df["business_name"]
        .apply(normalize_text)
    )

    df["address_norm"] = (
        df["business_address"]
        .apply(normalize_text)
    )

    df["country_norm"] = (
        df["country"]
        .apply(normalize_text)
    )

    return df


print("===================================")
print("RESTORING TEST DATA")
print("===================================")

test_s1 = preprocess_dataframe(
    pd.read_csv(
        TEST_PATH + "test_source1.tsv",
        sep="\t",
        dtype=str
    )
)

test_s2 = preprocess_dataframe(
    pd.read_csv(
        TEST_PATH + "test_source2.tsv",
        sep="\t",
        dtype=str
    )
)

test_s3 = preprocess_dataframe(
    pd.read_csv(
        TEST_PATH + "test_source3.tsv",
        sep="\t",
        dtype=str
    )
)

test_pool = pd.concat(
    [test_s2, test_s3],
    ignore_index=True
)

print("S1:", test_s1.shape)
print("S2:", test_s2.shape)
print("S3:", test_s3.shape)
print("Pool:", test_pool.shape)


# ------------------------------------------
# Exact-name lookup
# ------------------------------------------

print("\nBuilding exact-name index...")

test_exact_lookup = {}

for idx, name in enumerate(
    test_pool["name_norm"].values
):

    if name:
        test_exact_lookup.setdefault(
            name,
            []
        ).append(idx)


# ------------------------------------------
# Prefix lookup
# ------------------------------------------

print("Building prefix index...")

test_prefix_lookup = {}

for idx, name in enumerate(
    test_pool["name_norm"].values
):

    if name:

        prefix = name[:4]

        if prefix:
            test_prefix_lookup.setdefault(
                prefix,
                []
            ).append(idx)


print()
print("===================================")
print("TEST DATA READY")
print("===================================")
print("S1:", test_s1.shape)
print("Pool:", test_pool.shape)
print(
    "Exact groups:",
    len(test_exact_lookup)
)
print(
    "Prefix groups:",
    len(test_prefix_lookup)
)
print(
    "Model:",
    type(model).__name__
)
print(
    "Threshold:",
    BEST_THRESHOLD
)

RESTORING TEST DATA
S1: (1732544, 7)
S2: (4887273, 7)
S3: (4771003, 7)
Pool: (9658276, 7)

Building exact-name index...
Building prefix index...

TEST DATA READY
S1: (1732544, 7)
Pool: (9658276, 7)
Exact groups: 6858570
Prefix groups: 131564


NameError: name 'model' is not defined

In [3]:
# ==========================================
# STEP 2B - PROPER LIGHTWEIGHT MODEL
# ==========================================

from rapidfuzz import fuzz
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, fbeta_score
import pandas as pd
import numpy as np
import re
import gc

print("===================================")
print("STEP 2B - TRAINING MODEL")
print("===================================")

TRAIN_S1_LIMIT = 2000
MAX_CANDIDATES_PER_S1 = 8

# ------------------------------------------
# Load training S1 + ground truth
# ------------------------------------------

TRAIN_PATH = "../dataset/train/"

tr_s1 = pd.read_csv(
    TRAIN_PATH + "train_source1.tsv",
    sep="\t",
    dtype=str
)

tr_gt = pd.read_csv(
    TRAIN_PATH + "train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

tr_s1 = preprocess_dataframe(
    tr_s1.iloc[:TRAIN_S1_LIMIT]
)


def parse_gt(x):

    if pd.isna(x):
        return []

    x = str(x).strip()

    if not x:
        return []

    return [
        v.strip()
        for v in x.split(",")
        if v.strip()
    ]


tr_gt["matched_list"] = (
    tr_gt["matched_entity_ids"]
    .apply(parse_gt)
)

gt_lookup = dict(
    zip(
        tr_gt["source1_entity_id"],
        tr_gt["matched_list"]
    )
)


# ------------------------------------------
# Training S2/S3
# ------------------------------------------

tr_s2 = preprocess_dataframe(
    pd.read_csv(
        TRAIN_PATH + "train_source2.tsv",
        sep="\t",
        dtype=str
    )
)

tr_s3 = preprocess_dataframe(
    pd.read_csv(
        TRAIN_PATH + "train_source3.tsv",
        sep="\t",
        dtype=str
    )
)

tr_pool = pd.concat(
    [tr_s2, tr_s3],
    ignore_index=True
)

print(
    "Training pool:",
    tr_pool.shape
)


# ------------------------------------------
# Exact name lookup ONLY
# ------------------------------------------

tr_exact = {}

for idx, name in enumerate(
    tr_pool["name_norm"].values
):

    if name:
        tr_exact.setdefault(
            name,
            []
        ).append(idx)


# ------------------------------------------
# Build controlled pairs
# ------------------------------------------

pairs = []

for counter, row in enumerate(
    tr_s1.itertuples(index=False),
    start=1
):

    s1_id = row.entity_id
    name = row.name_norm
    country = row.country_norm

    true_ids = set(
        gt_lookup.get(
            s1_id,
            []
        )
    )

    candidates = []

    # Exact-name candidates
    if name:

        for idx in tr_exact.get(
            name,
            []
        ):

            if (
                tr_pool.iloc[idx]["country_norm"]
                == country
            ):
                candidates.append(idx)

    # Keep all true matches if available
    true_indices = []

    for idx in candidates:

        if (
            tr_pool.iloc[idx]["entity_id"]
            in true_ids
        ):
            true_indices.append(idx)

    # Add only a few negatives
    negative_indices = [
        idx for idx in candidates
        if (
            tr_pool.iloc[idx]["entity_id"]
            not in true_ids
        )
    ][:MAX_CANDIDATES_PER_S1]

    selected = (
        true_indices +
        negative_indices
    )

    for idx in selected:

        pairs.append({
            "s1_id": s1_id,
            "candidate_id":
                tr_pool.iloc[idx]["entity_id"],
            "label":
                int(
                    tr_pool.iloc[idx]["entity_id"]
                    in true_ids
                )
        })

    if counter % 500 == 0:

        print(
            f"Processed "
            f"{counter:,}/"
            f"{TRAIN_S1_LIMIT:,}"
        )


pairs_df = pd.DataFrame(pairs)

print()
print(
    "Pairs:",
    len(pairs_df)
)

print(
    "Positive:",
    int(pairs_df["label"].sum())
)

print(
    "Negative:",
    int(
        (pairs_df["label"] == 0).sum()
    )
)


# ------------------------------------------
# Feature generation
# ------------------------------------------

print("\nBuilding features...")

tr_s1_lookup = tr_s1.set_index(
    "entity_id"
)

tr_pool_lookup = tr_pool.set_index(
    "entity_id"
)

features = []


for row in pairs_df.itertuples(
    index=False
):

    a = tr_s1_lookup.loc[row.s1_id]
    b = tr_pool_lookup.loc[row.candidate_id]

    n1 = a["name_norm"]
    n2 = b["name_norm"]

    ad1 = a["address_norm"]
    ad2 = b["address_norm"]

    c1 = a["country_norm"]
    c2 = b["country_norm"]

    nums1 = set(
        re.findall(
            r"\b\d+\b",
            ad1
        )
    )

    nums2 = set(
        re.findall(
            r"\b\d+\b",
            ad2
        )
    )

    if nums1 and nums2:

        num_overlap = (
            len(nums1 & nums2)
            /
            len(nums1 | nums2)
        )

    else:

        num_overlap = 0.0

    features.append([
        fuzz.ratio(n1, n2) / 100,
        fuzz.token_set_ratio(n1, n2) / 100,
        fuzz.token_sort_ratio(n1, n2) / 100,
        fuzz.partial_ratio(n1, n2) / 100,
        int(
            n1 == n2 and n1 != ""
        ),
        fuzz.ratio(ad1, ad2) / 100,
        fuzz.token_set_ratio(ad1, ad2) / 100,
        fuzz.partial_ratio(ad1, ad2) / 100,
        num_overlap,
        int(c1 == c2),
        abs(len(n1) - len(n2)),
        abs(len(ad1) - len(ad2))
    ])


feature_columns = [
    "name_ratio",
    "name_token_set",
    "name_token_sort",
    "name_partial",
    "exact_name",
    "address_ratio",
    "address_token_set",
    "address_partial",
    "address_num_overlap",
    "country_match",
    "name_length_diff",
    "address_length_diff"
]

X = pd.DataFrame(
    features,
    columns=feature_columns
)

y = pairs_df["label"]


# ------------------------------------------
# Train / validation
# ------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Random Forest...")

model.fit(
    X_train,
    y_train
)


# ------------------------------------------
# Threshold
# ------------------------------------------

val_prob = model.predict_proba(
    X_val
)[:, 1]

best_threshold = 0.75

val_pred = (
    val_prob >= best_threshold
).astype(int)

print()
print("===================================")
print("STEP 2B COMPLETE")
print("===================================")
print("Feature matrix:", X.shape)
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print(
    "Validation precision:",
    round(
        precision_score(
            y_val,
            val_pred,
            zero_division=0
        ),
        4
    )
)
print(
    "Validation recall:",
    round(
        recall_score(
            y_val,
            val_pred,
            zero_division=0
        ),
        4
    )
)
print(
    "Validation F0.5:",
    round(
        fbeta_score(
            y_val,
            val_pred,
            beta=0.5,
            zero_division=0
        ),
        4
    )
)

BEST_THRESHOLD = 0.75

print(
    "Final threshold:",
    BEST_THRESHOLD
)

STEP 2B - TRAINING MODEL
Training pool: (10320219, 7)
Processed 500/2,000
Processed 1,000/2,000
Processed 1,500/2,000
Processed 2,000/2,000

Pairs: 5807
Positive: 1820
Negative: 3987

Building features...

Training Random Forest...

STEP 2B COMPLETE
Feature matrix: (5807, 12)
Training samples: 4645
Validation samples: 1162
Validation precision: 0.9743
Validation recall: 0.9368
Validation F0.5: 0.9666
Final threshold: 0.75


In [6]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

print("Warnings disabled")

Warnings disabled


In [ ]:
# ============================================================
# STEP 3 - BULK FAST INFERENCE
# ============================================================

import os
import time
import gc
import pandas as pd
import numpy as np

MATCHING_OUT = "../output/matching_results.tsv"
CANDIDATE_OUT = "../output/candidate_pairs.tsv"

start_time = time.time()

print("===================================")
print("STEP 3 - BULK FAST INFERENCE")
print("===================================")

# ============================================================
# 1. PREPARE TEST POOL
# ============================================================

print("Preparing pool...")

pool = test_pool[
    [
        "entity_id",
        "name_norm",
        "country_norm"
    ]
].copy()

# Remove empty names because they are not useful for exact matching
pool = pool[
    pool["name_norm"].notna()
    &
    (pool["name_norm"] != "")
].copy()

print("Usable pool:", f"{len(pool):,}")

# ============================================================
# 2. CREATE EXACT NAME + COUNTRY KEY
# ============================================================

pool["match_key"] = (
    pool["country_norm"].astype(str)
    + "|||"
    + pool["name_norm"].astype(str)
)

# ============================================================
# 3. GROUP IDS BY KEY
# ============================================================

print("Building exact lookup...")

exact_groups = (
    pool.groupby("match_key")["entity_id"]
    .agg(list)
)

print("Unique exact keys:", f"{len(exact_groups):,}")

# ============================================================
# 4. PREPARE TEST S1
# ============================================================

s1 = test_s1[
    [
        "entity_id",
        "name_norm",
        "country_norm"
    ]
].copy()

s1["match_key"] = (
    s1["country_norm"].astype(str)
    + "|||"
    + s1["name_norm"].astype(str)
)

# Empty names should never generate matches
empty_name_mask = (
    s1["name_norm"].isna()
    |
    (s1["name_norm"] == "")
)

# ============================================================
# 5. BULK LOOKUP
# ============================================================

print("Performing bulk lookup...")

s1["matched_list"] = (
    s1["match_key"]
    .map(exact_groups)
)

# ============================================================
# 6. CONVERT MISSING LOOKUPS TO EMPTY
# ============================================================

s1["matched_list"] = s1["matched_list"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Empty source names = no match
s1.loc[
    empty_name_mask,
    "matched_list"
] = [[] for _ in range(empty_name_mask.sum())]

# ============================================================
# 7. CREATE MATCHING OUTPUT
# ============================================================

print("Creating matching_results.tsv...")

matching_output = pd.DataFrame({
    "source1_entity_id": s1["entity_id"],
    "matched_entity_ids": s1["matched_list"].apply(
        lambda x: ",".join(map(str, x))
    )
})

# ============================================================
# 8. CANDIDATE OUTPUT
# ============================================================
#
# For this ultra-fast version, the final candidate set is
# the exact normalized-name + country candidate set.
#
# Every predicted match is therefore guaranteed to be inside
# the candidate set.
# ============================================================

print("Creating candidate_pairs.tsv...")

candidate_output = pd.DataFrame({
    "source1_entity_id": s1["entity_id"],
    "candidate_entity_ids": s1["matched_list"].apply(
        lambda x: ",".join(map(str, x))
    )
})

# ============================================================
# 9. WRITE FILES
# ============================================================

os.makedirs("../output", exist_ok=True)

matching_output.to_csv(
    MATCHING_OUT,
    sep="\t",
    index=False
)

candidate_output.to_csv(
    CANDIDATE_OUT,
    sep="\t",
    index=False
)

# ============================================================
# 10. CLEAN MEMORY
# ============================================================

del pool
del exact_groups
del s1

gc.collect()

# ============================================================
# 11. OUTPUT CHECK
# ============================================================

elapsed = (time.time() - start_time) / 60

print()
print("===================================")
print("STEP 3 COMPLETE")
print("===================================")

print(
    "matching_results.tsv:",
    MATCHING_OUT
)

print(
    "candidate_pairs.tsv:",
    CANDIDATE_OUT
)

print(
    "Rows:",
    len(matching_output)
)

print(
    "Non-empty matches:",
    (
        matching_output["matched_entity_ids"]
        .ne("")
        .sum()
    )
)

print(
    "Empty matches:",
    (
        matching_output["matched_entity_ids"]
        .eq("")
        .sum()
    )
)

print(
    "Time:",
    round(elapsed, 2),
    "minutes"
)

print()
print("First 5 rows:")
display(matching_output.head())

STEP 3 - BULK FAST INFERENCE
Preparing pool...
Usable pool: 8,868,506
Building exact lookup...


In [1]:
# ============================================================
# STEP 3 - BULK FAST INFERENCE
# ============================================================

import os
import time
import gc
import pandas as pd
import numpy as np

MATCHING_OUT = "../output/matching_results.tsv"
CANDIDATE_OUT = "../output/candidate_pairs.tsv"

start_time = time.time()

print("===================================")
print("STEP 3 - BULK FAST INFERENCE")
print("===================================")

# ============================================================
# 1. PREPARE TEST POOL
# ============================================================

print("Preparing pool...")

pool = test_pool[
    [
        "entity_id",
        "name_norm",
        "country_norm"
    ]
].copy()

# Remove empty names because they are not useful for exact matching
pool = pool[
    pool["name_norm"].notna()
    &
    (pool["name_norm"] != "")
].copy()

print("Usable pool:", f"{len(pool):,}")

# ============================================================
# 2. CREATE EXACT NAME + COUNTRY KEY
# ============================================================

pool["match_key"] = (
    pool["country_norm"].astype(str)
    + "|||"
    + pool["name_norm"].astype(str)
)

# ============================================================
# 3. GROUP IDS BY KEY
# ============================================================

print("Building exact lookup...")

exact_groups = (
    pool.groupby("match_key")["entity_id"]
    .agg(list)
)

print("Unique exact keys:", f"{len(exact_groups):,}")

# ============================================================
# 4. PREPARE TEST S1
# ============================================================

s1 = test_s1[
    [
        "entity_id",
        "name_norm",
        "country_norm"
    ]
].copy()

s1["match_key"] = (
    s1["country_norm"].astype(str)
    + "|||"
    + s1["name_norm"].astype(str)
)

# Empty names should never generate matches
empty_name_mask = (
    s1["name_norm"].isna()
    |
    (s1["name_norm"] == "")
)

# ============================================================
# 5. BULK LOOKUP
# ============================================================

print("Performing bulk lookup...")

s1["matched_list"] = (
    s1["match_key"]
    .map(exact_groups)
)

# ============================================================
# 6. CONVERT MISSING LOOKUPS TO EMPTY
# ============================================================

s1["matched_list"] = s1["matched_list"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Empty source names = no match
s1.loc[
    empty_name_mask,
    "matched_list"
] = [[] for _ in range(empty_name_mask.sum())]

# ============================================================
# 7. CREATE MATCHING OUTPUT
# ============================================================

print("Creating matching_results.tsv...")

matching_output = pd.DataFrame({
    "source1_entity_id": s1["entity_id"],
    "matched_entity_ids": s1["matched_list"].apply(
        lambda x: ",".join(map(str, x))
    )
})

# ============================================================
# 8. CANDIDATE OUTPUT
# ============================================================
#
# For this ultra-fast version, the final candidate set is
# the exact normalized-name + country candidate set.
#
# Every predicted match is therefore guaranteed to be inside
# the candidate set.
# ============================================================

print("Creating candidate_pairs.tsv...")

candidate_output = pd.DataFrame({
    "source1_entity_id": s1["entity_id"],
    "candidate_entity_ids": s1["matched_list"].apply(
        lambda x: ",".join(map(str, x))
    )
})

# ============================================================
# 9. WRITE FILES
# ============================================================

os.makedirs("../output", exist_ok=True)

matching_output.to_csv(
    MATCHING_OUT,
    sep="\t",
    index=False
)

candidate_output.to_csv(
    CANDIDATE_OUT,
    sep="\t",
    index=False
)

# ============================================================
# 10. CLEAN MEMORY
# ============================================================

del pool
del exact_groups
del s1

gc.collect()

# ============================================================
# 11. OUTPUT CHECK
# ============================================================

elapsed = (time.time() - start_time) / 60

print()
print("===================================")
print("STEP 3 COMPLETE")
print("===================================")

print(
    "matching_results.tsv:",
    MATCHING_OUT
)

print(
    "candidate_pairs.tsv:",
    CANDIDATE_OUT
)

print(
    "Rows:",
    len(matching_output)
)

print(
    "Non-empty matches:",
    (
        matching_output["matched_entity_ids"]
        .ne("")
        .sum()
    )
)

print(
    "Empty matches:",
    (
        matching_output["matched_entity_ids"]
        .eq("")
        .sum()
    )
)

print(
    "Time:",
    round(elapsed, 2),
    "minutes"
)

print()
print("First 5 rows:")
display(matching_output.head())

STEP 3 - BULK FAST INFERENCE
Preparing pool...


NameError: name 'test_pool' is not defined

In [2]:

# ==========================================
# RECOVERY CELL - RESTORE TEST DATA
# ==========================================

import os
import re
import glob
import gc
import unicodedata
import pandas as pd
import numpy as np

TEST_PATH = "../dataset/test/"

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_dataframe(df):
    df = df.copy()
    df["name_norm"] = df["business_name"].apply(normalize_text)
    df["address_norm"] = df["business_address"].apply(normalize_text)
    df["country_norm"] = df["country"].apply(normalize_text)
    return df

# Find the three test TSV files
files = glob.glob(os.path.join(TEST_PATH, "**", "*.tsv"), recursive=True)
print("TSV files found:")
for f in files:
    print(os.path.basename(f))

def find_source(n):
    patterns = [
        rf"source[_ -]?{n}(?:[_ -]|\.|$)",
        rf"\bs{n}(?:[_ -]|\.|$)"
    ]
    matches = [
        f for f in files
        if any(re.search(p, os.path.basename(f).lower()) for p in patterns)
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Could not uniquely identify Source {n}. "
            f"Please check the filenames printed above."
        )
    return matches[0]

s1_file = find_source(1)
s2_file = find_source(2)
s3_file = find_source(3)

print("\nLoading test sources...")
test_s1 = pd.read_csv(s1_file, sep="\t")
test_s2 = pd.read_csv(s2_file, sep="\t")
test_s3 = pd.read_csv(s3_file, sep="\t")

print("Preprocessing...")
test_s1 = preprocess_dataframe(test_s1)
test_s2 = preprocess_dataframe(test_s2)
test_s3 = preprocess_dataframe(test_s3)

print("Combining candidate pool...")
test_pool = pd.concat(
    [test_s2, test_s3],
    ignore_index=True
)

print("\n===================================")
print("TEST DATA RESTORED")
print("===================================")
print("S1:", f"{len(test_s1):,}")
print("S2:", f"{len(test_s2):,}")
print("S3:", f"{len(test_s3):,}")
print("Pool:", f"{len(test_pool):,}")
print("Columns:", test_pool.columns.tolist())

gc.collect()

TSV files found:
test_source2.tsv
test_source3.tsv
test_source1.tsv

Loading test sources...
Preprocessing...
Combining candidate pool...

TEST DATA RESTORED
S1: 1,732,544
S2: 4,887,273
S3: 4,771,003
Pool: 9,658,276
Columns: ['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'address_norm', 'country_norm']


564

In [3]:
# ==========================================
# STEP 3: FAST EXACT-MATCH INFERENCE
# ==========================================

import os
import gc
import pandas as pd

# Output folder
OUTPUT_DIR = "../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Required columns
s1 = test_s1[["entity_id", "name_norm", "country_norm"]].copy()
pool = test_pool[["entity_id", "name_norm", "country_norm"]].copy()

# Remove records with empty business names
s1 = s1[s1["name_norm"].ne("")].copy()
pool = pool[pool["name_norm"].ne("")].copy()

# Build exact name + country lookup
print("Building exact-match index...")

pool["match_key"] = (
    pool["country_norm"].astype(str) + "||" +
    pool["name_norm"].astype(str)
)

s1["match_key"] = (
    s1["country_norm"].astype(str) + "||" +
    s1["name_norm"].astype(str)
)

# Group all matching pool entity IDs by key
match_lookup = (
    pool.groupby("match_key", sort=False)["entity_id"]
    .agg(list)
    .to_dict()
)

print("Index created:", len(match_lookup))

# Generate matching and candidate outputs
print("Generating predictions...")

matching_rows = []
candidate_rows = []

for start in range(0, len(s1), 100_000):
    batch = s1.iloc[start:start + 100_000]

    for entity_id, key in zip(batch["entity_id"], batch["match_key"]):
        ids = match_lookup.get(key, [])

        # Exclude the S1 entity itself, if present
        ids = [str(x) for x in ids if str(x) != str(entity_id)]

        matching_rows.append({
            "source1_entity_id": entity_id,
            "matched_entity_ids": ",".join(ids)
        })

        candidate_rows.append({
            "source1_entity_id": entity_id,
            "candidate_entity_ids": ",".join(ids)
        })

    print(f"Processed {min(start + 100_000, len(s1)):,} / {len(s1):,}")

# Create output dataframes
matching_df = pd.DataFrame(
    matching_rows,
    columns=["source1_entity_id", "matched_entity_ids"]
)

candidate_df = pd.DataFrame(
    candidate_rows,
    columns=["source1_entity_id", "candidate_entity_ids"]
)

# Save required files
matching_path = os.path.join(OUTPUT_DIR, "matching_results.tsv")
candidate_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")

matching_df.to_csv(matching_path, sep="\t", index=False)
candidate_df.to_csv(candidate_path, sep="\t", index=False)

print("\n====================================")
print("EXACT-MATCH INFERENCE COMPLETED")
print("====================================")
print("Matching file:", matching_path)
print("Candidate file:", candidate_path)
print("Matching rows:", f"{len(matching_df):,}")
print("Candidate rows:", f"{len(candidate_df):,}")
print("Predicted non-empty matches:",
      f"{matching_df['matched_entity_ids'].ne('').sum():,}")

del pool, s1, match_lookup, matching_rows, candidate_rows
gc.collect()

Building exact-match index...
Index created: 6872632
Generating predictions...
Processed 100,000 / 1,732,544
Processed 200,000 / 1,732,544
Processed 300,000 / 1,732,544
Processed 400,000 / 1,732,544
Processed 500,000 / 1,732,544
Processed 600,000 / 1,732,544
Processed 700,000 / 1,732,544
Processed 800,000 / 1,732,544
Processed 900,000 / 1,732,544
Processed 1,000,000 / 1,732,544
Processed 1,100,000 / 1,732,544
Processed 1,200,000 / 1,732,544
Processed 1,300,000 / 1,732,544
Processed 1,400,000 / 1,732,544
Processed 1,500,000 / 1,732,544
Processed 1,600,000 / 1,732,544
Processed 1,700,000 / 1,732,544
Processed 1,732,544 / 1,732,544

EXACT-MATCH INFERENCE COMPLETED
Matching file: ../output/matching_results.tsv
Candidate file: ../output/candidate_pairs.tsv
Matching rows: 1,732,544
Candidate rows: 1,732,544
Predicted non-empty matches: 1,296,366


0